# نسخة قياس أداء Blockchain — Douban Movie

هذه نسخة منفصلة من الكود المعتمد. لم يتم تغيير منطق التصويت أو اختيار $A_k$ أو العقد الذكي أو عتبة الأغلبية. أضيف فقط تسجيل Gas، زمن المعاملة، بيانات الكلفة، وملخصات الأداء. **ابدأ على Ganache جديدة، وأعد تنفيذ Wallet Mapping قبل نشر العقد** لأن عناوين الحسابات تتغير عند إعادة تشغيل Ganache بدون قاعدة بيانات محفوظة.


### الخطوة 1
### اعلان الترستدنود 1

هذا الكود لا يعيد اختيار 
A_k
، وإنما يعمل فقط على تجهيز ملفات مرحلة الإعلان:

ينشئ مجلدات مشروع الإعلان.
يحوّل ملف interest_membership.csv
إلى ملف community_members.csv.
يقرأ ملف نتائج اختيار القائد.
ينشئ ملف ak_chain_announcements.csv.
ينشئ ملف JSON لكل مجتمع يحتوي A_k
 و result_hash.
يحسب عدد التأكيدات المطلوبة لكل مجتمع حسب الأغلبية البسيطة.

In [1]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Prepare Announcement Stage for Trusted Authorized Node A_k
# =====================================================================================
#
# الهدف:
# هذا السكربت لا يعيد اختيار A_k.
# هو يأخذ نتائج الاختيار السابقة من مرحلة التصويت، ثم يجهز ملفات مرحلة الإعلان:
#
# 1) إنشاء هيكل مجلدات مرحلة الإعلان.
# 2) إنشاء ملف community_members.csv المطلوب لتحميل أعضاء المجتمع إلى العقد الذكي.
# 3) إنشاء ملف ak_chain_announcements.csv المطلوب لإعلان A_k على البلوكشين.
# 4) إنشاء ملف JSON لكل مجتمع يحتوي تفاصيل الإعلان و result_hash.
# 5) إنشاء تقرير تحقق مختصر validation_report.txt.
#
# المسار المعتمد لمرحلة الإعلان:
# D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان
#
# =====================================================================================

from __future__ import annotations

import json
import hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd


# =====================================================================================
# Cell 0 — Main Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

VOTING_RUN_ID = "RUN_20260309_031845"

VOTING_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\2 التصويت\OP"
) / VOTING_RUN_ID

INTEREST_MEMBERSHIP_CSV = (
    VOTING_ROOT
    / "outputs"
    / "step1_interest"
    / "interest_membership.csv"
)

SUMMARY_CSV = (
    VOTING_ROOT
    / "outputs"
    / "scenario1_global_friend_distance_sumdist_no_fallback"
    / "final"
    / "scenario1_global_friend_sumdist_no_fallback_summary.csv"
)

VOTE_SCENARIO_DIR = (
    VOTING_ROOT
    / "outputs"
    / "scenario1_global_friend_distance_sumdist_no_fallback"
)


# =====================================================================================
# Cell 1 — Announcement Project Folders
# =====================================================================================

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

COMMUNITIES_DIR = DATA_DIR / "communities"
BLOCKCHAIN_DIR = DATA_DIR / "blockchain"
CHAIN_ANNOUNCEMENTS_DIR = DATA_DIR / "chain_announcements"

CONTRACTS_DIR = ANNOUNCEMENT_ROOT / "contracts"
CONTRACTS_BUILD_DIR = CONTRACTS_DIR / "build"

SCRIPTS_DIR = ANNOUNCEMENT_ROOT / "scripts"
OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

COMMUNITY_MEMBERS_OUT = COMMUNITIES_DIR / "community_members.csv"
AK_ANNOUNCEMENTS_OUT = CHAIN_ANNOUNCEMENTS_DIR / "ak_chain_announcements.csv"
VALIDATION_REPORT_OUT = LOGS_DIR / "validation_report.txt"


# =====================================================================================
# Cell 2 — General Configuration
# =====================================================================================

ALGORITHM_NAME = "Scenario1_GlobalNearest_FriendDistance_MinSum_NoFallback"

INCLUDE_DETAIL_FILE_HASHES = True


# =====================================================================================
# Cell 3 — Helper Functions
# =====================================================================================

def ensure_dirs() -> None:
    """
    إنشاء مجلدات مشروع الإعلان.
    """
    folders = [
        ANNOUNCEMENT_ROOT,
        DATA_DIR,
        COMMUNITIES_DIR,
        BLOCKCHAIN_DIR,
        CHAIN_ANNOUNCEMENTS_DIR,
        CONTRACTS_DIR,
        CONTRACTS_BUILD_DIR,
        SCRIPTS_DIR,
        OUTPUTS_DIR,
        LOGS_DIR,
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)


def ensure_file_exists(path: Path, label: str) -> None:
    """
    التحقق من وجود ملف مطلوب.
    """
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def stable_json_hash(payload: dict) -> str:
    """
    بناء SHA-256 hash ثابت من محتوى منظم.
    """
    normalized = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def file_sha256(path: Path) -> str | None:
    """
    حساب SHA-256 لأي ملف تفصيلي إذا كان موجودًا.
    إذا لم يكن موجودًا، نرجع None.
    """
    if not path.exists():
        return None

    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def clean_int(value, field_name: str) -> int:
    """
    تحويل قيمة إلى int مع رسالة خطأ واضحة.
    """
    if pd.isna(value):
        raise ValueError(f"[ERROR] Missing value in field: {field_name}")

    return int(value)


def safe_str(value) -> str:
    """
    تحويل آمن إلى string.
    """
    if pd.isna(value):
        return ""
    return str(value)


def safe_bool(value) -> bool:
    """
    تحويل آمن إلى bool، لأن بعض ملفات CSV قد تقرأ True/False كنصوص.
    """
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no"}:
        return False

    return bool(value)


# =====================================================================================
# Cell 4 — Build community_members.csv
# =====================================================================================

def build_community_members_file() -> pd.DataFrame:
    """
    الهدف:
    تحويل ملف interest_membership.csv الناتج من مرحلة التصويت إلى ملف مناسب للبلوكشين.

    الملف الأصلي المتوقع:
        interest_comm_id,User

    الملف الناتج:
        user_id,community_id

    لماذا نحتاجه؟
    لأن العقد الذكي يحتاج معرفة أعضاء كل community حتى يقبل announce/confirm فقط من الأعضاء.
    """
    ensure_file_exists(INTEREST_MEMBERSHIP_CSV, "interest_membership.csv")

    df = pd.read_csv(INTEREST_MEMBERSHIP_CSV)

    expected_cols = {"interest_comm_id", "User"}
    missing = expected_cols - set(df.columns)

    if missing:
        raise ValueError(
            f"[ERROR] interest_membership.csv missing columns: {missing}. "
            f"Found columns: {list(df.columns)}"
        )

    out = df.rename(
        columns={
            "User": "user_id",
            "interest_comm_id": "community_id"
        }
    )[["user_id", "community_id"]].copy()

    out = out.dropna(subset=["user_id", "community_id"])
    out["user_id"] = out["user_id"].astype(int)
    out["community_id"] = out["community_id"].astype(int)

    out = (
        out.drop_duplicates()
        .sort_values(["community_id", "user_id"])
        .reset_index(drop=True)
    )

    out.to_csv(COMMUNITY_MEMBERS_OUT, index=False, encoding="utf-8-sig")

    return out


# =====================================================================================
# Cell 5 — Build A_k Announcement Files
# =====================================================================================

def build_ak_announcement_files(community_members: pd.DataFrame) -> pd.DataFrame:
    """
    الهدف:
    قراءة ملف summary الناتج من خوارزمية اختيار A_k، ثم تجهيز ملف الإعلان.

    الملف الأصلي:
        scenario1_global_friend_sumdist_no_fallback_summary.csv

    الملف الناتج:
        ak_chain_announcements.csv

    يحتوي:
        community_id
        A_k
        result_hash
        required_confirmations
        decision_basis
        interest_size
    """
    ensure_file_exists(SUMMARY_CSV, "A_k summary CSV")

    df_summary = pd.read_csv(SUMMARY_CSV)

    required_summary_cols = {
        "interest_comm_id",
        "interest_size",
        "nominating_voters",
        "abstain_count",
        "nomination_coverage",
        "unique_nominees",
        "leader",
        "leader_sum_dist",
        "leader_coverage_count",
        "leader_coverage_ratio",
        "leader_nominated_by_count",
        "leader_in_I",
        "decision_basis",
        "tie_group_size",
    }

    missing = required_summary_cols - set(df_summary.columns)

    if missing:
        raise ValueError(
            f"[ERROR] summary CSV missing columns: {missing}. "
            f"Found columns: {list(df_summary.columns)}"
        )

    rows = []

    community_sizes_from_members = (
        community_members
        .groupby("community_id")["user_id"]
        .nunique()
        .to_dict()
    )

    for _, r in df_summary.iterrows():
        community_id = clean_int(r["interest_comm_id"], "interest_comm_id")

        if pd.isna(r["leader"]):
            print(f"[WARNING] Community {community_id} has no leader. Skipped.")
            continue

        Ak = clean_int(r["leader"], "leader")

        interest_size_summary = clean_int(r["interest_size"], "interest_size")

        interest_size_members = int(
            community_sizes_from_members.get(community_id, 0)
        )

        community_size_for_chain = interest_size_members

        required_confirmations = (community_size_for_chain // 2) + 1

        nomination_file = (
            VOTE_SCENARIO_DIR
            / "nominations_per_interest"
            / f"nominations_global_friend_comm_{community_id}.csv"
        )

        candidate_file = (
            VOTE_SCENARIO_DIR
            / "candidate_scores_per_interest"
            / f"candidate_scores_comm_{community_id}.csv"
        )

        candidate_scores_hash = None
        nominations_hash = None

        if INCLUDE_DETAIL_FILE_HASHES:
            candidate_scores_hash = file_sha256(candidate_file)
            nominations_hash = file_sha256(nomination_file)

        payload = {
            "algorithm": ALGORITHM_NAME,
            "voting_run_id": VOTING_RUN_ID,
            "community_id": community_id,
            "A_k": Ak,

            "interest_size_from_summary": interest_size_summary,
            "interest_size_from_membership_file": interest_size_members,
            "community_size_used_on_chain": community_size_for_chain,
            "required_confirmations": required_confirmations,

            "nominating_voters": clean_int(r["nominating_voters"], "nominating_voters"),
            "abstain_count": clean_int(r["abstain_count"], "abstain_count"),
            "nomination_coverage": float(r["nomination_coverage"]),
            "unique_nominees": clean_int(r["unique_nominees"], "unique_nominees"),
            "leader_sum_dist": float(r["leader_sum_dist"]),
            "leader_coverage_count": clean_int(r["leader_coverage_count"], "leader_coverage_count"),
            "leader_coverage_ratio": float(r["leader_coverage_ratio"]),
            "leader_nominated_by_count": clean_int(
                r["leader_nominated_by_count"],
                "leader_nominated_by_count"
            ),
            "leader_in_I": safe_bool(r["leader_in_I"]),
            "decision_basis": safe_str(r["decision_basis"]),
            "tie_group_size": clean_int(r["tie_group_size"], "tie_group_size"),

            "candidate_scores_sha256": candidate_scores_hash,
            "nominations_sha256": nominations_hash,
        }

        result_hash = stable_json_hash(payload)

        payload_with_hash = {
            **payload,
            "result_hash": result_hash,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        }

        json_path = (
            CHAIN_ANNOUNCEMENTS_DIR
            / f"ak_announcement_comm_{community_id}.json"
        )

        json_path.write_text(
            json.dumps(payload_with_hash, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

        rows.append({
            "community_id": community_id,
            "A_k": Ak,
            "result_hash": result_hash,
            "required_confirmations": required_confirmations,
            "community_size_used_on_chain": community_size_for_chain,
            "interest_size_from_summary": interest_size_summary,
            "interest_size_from_membership_file": interest_size_members,
            "nominating_voters": clean_int(r["nominating_voters"], "nominating_voters"),
            "abstain_count": clean_int(r["abstain_count"], "abstain_count"),
            "unique_nominees": clean_int(r["unique_nominees"], "unique_nominees"),
            "decision_basis": safe_str(r["decision_basis"]),
            "leader_in_I": safe_bool(r["leader_in_I"]),
            "json_file": str(json_path),
        })

    df_out = (
        pd.DataFrame(rows)
        .sort_values("community_id")
        .reset_index(drop=True)
    )

    df_out.to_csv(AK_ANNOUNCEMENTS_OUT, index=False, encoding="utf-8-sig")

    return df_out


# =====================================================================================
# Cell 6 — Validation Report
# =====================================================================================

def write_validation_report(
    community_members: pd.DataFrame,
    ak_announcements: pd.DataFrame
) -> None:
    """
    إنشاء تقرير مختصر حتى نراجع صحة الملفات قبل الانتقال إلى Ganache/Remix.
    """
    lines = []

    lines.append("=" * 90)
    lines.append("Announcement Stage Validation Report")
    lines.append("=" * 90)
    lines.append("")

    lines.append("[Paths]")
    lines.append(f"ANNOUNCEMENT_ROOT       : {ANNOUNCEMENT_ROOT}")
    lines.append(f"VOTING_ROOT             : {VOTING_ROOT}")
    lines.append(f"INTEREST_MEMBERSHIP_CSV : {INTEREST_MEMBERSHIP_CSV}")
    lines.append(f"SUMMARY_CSV             : {SUMMARY_CSV}")
    lines.append("")

    lines.append("[Generated Files]")
    lines.append(f"COMMUNITY_MEMBERS_OUT   : {COMMUNITY_MEMBERS_OUT}")
    lines.append(f"AK_ANNOUNCEMENTS_OUT    : {AK_ANNOUNCEMENTS_OUT}")
    lines.append("")

    lines.append("[Community Membership Summary]")
    total_users = community_members["user_id"].nunique()
    total_rows = len(community_members)
    total_communities = community_members["community_id"].nunique()

    lines.append(f"Total rows              : {total_rows}")
    lines.append(f"Total unique users      : {total_users}")
    lines.append(f"Total communities       : {total_communities}")
    lines.append("")

    size_table = (
        community_members
        .groupby("community_id")["user_id"]
        .nunique()
        .reset_index(name="community_size")
        .sort_values("community_id")
    )

    lines.append("Community sizes:")
    lines.append(size_table.to_string(index=False))
    lines.append("")

    lines.append("[A_k Announcement Summary]")
    if len(ak_announcements) == 0:
        lines.append("No announcements generated.")
    else:
        cols = [
            "community_id",
            "A_k",
            "community_size_used_on_chain",
            "required_confirmations",
            "decision_basis",
            "leader_in_I",
        ]
        lines.append(ak_announcements[cols].to_string(index=False))

    lines.append("")
    lines.append("[Important Notes]")
    lines.append("1) This stage does NOT recompute A_k.")
    lines.append("2) A_k is read from the voting summary CSV.")
    lines.append("3) result_hash links the on-chain announcement with off-chain election outputs.")
    lines.append("4) required_confirmations = floor(community_size / 2) + 1.")
    lines.append("5) Ganache must contain enough accounts to represent confirming community members.")
    lines.append("")

    VALIDATION_REPORT_OUT.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


# =====================================================================================
# Cell 7 — Main
# =====================================================================================

def main() -> None:
    print("=" * 90)
    print("[START] Preparing Announcement Stage for A_k")
    print("=" * 90)

    print("\n[1] Creating announcement folders...")
    ensure_dirs()
    print("[OK] Folders created/verified.")
    print("[PATH] ANNOUNCEMENT_ROOT:", ANNOUNCEMENT_ROOT)

    print("\n[2] Checking required input files...")
    ensure_file_exists(INTEREST_MEMBERSHIP_CSV, "interest_membership.csv")
    ensure_file_exists(SUMMARY_CSV, "A_k summary CSV")
    print("[OK] Required input files exist.")

    print("\n[3] Building community_members.csv...")
    community_members = build_community_members_file()
    print("[OK] community_members.csv created.")
    print("[PATH]", COMMUNITY_MEMBERS_OUT)
    print("[INFO] Rows:", len(community_members))
    print("[INFO] Unique users:", community_members["user_id"].nunique())
    print("[INFO] Communities:", community_members["community_id"].nunique())

    print("\n[4] Building A_k announcement files...")
    ak_announcements = build_ak_announcement_files(community_members)
    print("[OK] ak_chain_announcements.csv created.")
    print("[PATH]", AK_ANNOUNCEMENTS_OUT)

    print("\n[5] Writing validation report...")
    write_validation_report(community_members, ak_announcements)
    print("[OK] validation_report.txt created.")
    print("[PATH]", VALIDATION_REPORT_OUT)

    print("\n[SUMMARY]")
    if len(ak_announcements) > 0:
        show_cols = [
            "community_id",
            "A_k",
            "community_size_used_on_chain",
            "required_confirmations",
            "decision_basis",
        ]
        print(ak_announcements[show_cols].to_string(index=False))
    else:
        print("No A_k announcements generated.")

    print("\n[DONE] Announcement preparation stage completed successfully.")
    print("=" * 90)


main()

[START] Preparing Announcement Stage for A_k

[1] Creating announcement folders...
[OK] Folders created/verified.
[PATH] ANNOUNCEMENT_ROOT: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان

[2] Checking required input files...
[OK] Required input files exist.

[3] Building community_members.csv...
[OK] community_members.csv created.
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\data\communities\community_members.csv
[INFO] Rows: 693
[INFO] Unique users: 693
[INFO] Communities: 6

[4] Building A_k announcement files...
[OK] ak_chain_announcements.csv created.
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\data\chain_announcements\ak_chain_announcements.csv

[5] Writing validation report...
[OK] validation_report.txt created.
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\outputs\logs\validation_report.txt

[SUMMARY]
 community_id  A_k  community_size_used_on_chain  required_confirmations deci

### الخطوة 2
### ربط الهاش الناتج من المرحلة السابقة مع عنوان كل مستخدم (نود اي دي)
يشترط الكناج شغال بواسطة هذا الامر اولا
ganache --wallet.totalAccounts=750 --chain.chainId=1337 --server.host=127.0.0.1 --server.port=7545 --wallet.defaultBalance=1000 --miner.blockGasLimit=80000000


معناه بالتفصيل:

الجزء	معناه
ganache	يشغل برنامج Ganache المحلي
--wallet.totalAccounts=750	ينشئ 750 حسابًا تجريبيًا
--chain.chainId=1337	يحدد رقم الشبكة بـ 1337
--server.host=127.0.0.1	يجعل الشبكة تعمل محليًا على جهازك فقط
--server.port=7545	يجعل Ganache يعمل على المنفذ 7545
--wallet.defaultBalance=1000	يعطي كل حساب 1000 ETH تجريبي
--miner.blockGasLimit=80000000	يرفع حد الغاز حتى نقدر نحمل أعضاء كثيرين للعقد

ليش نحتاج 750 حساب؟

لأن عندك:

693 user

In [2]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Build address_to_userid.csv for Ganache Wallet Mapping
# =====================================================================================
#
# الهدف:
# ربط user_id بعناوين Ganache حتى نستطيع لاحقًا تحميل أعضاء المجتمع إلى العقد الذكي
# والتحقق من أن msg.sender هو عضو فعلي في المجتمع.
#
# المخرجات:
# 1) address_to_userid.csv
# 2) owner_address.txt
# 3) wallet_mapping_report.txt
#
# =====================================================================================

from __future__ import annotations

from pathlib import Path
import pandas as pd
from web3 import Web3


# =====================================================================================
# Cell 0 — Paths and RPC
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

COMMUNITY_MEMBERS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "communities"
    / "community_members.csv"
)

AK_ANNOUNCEMENTS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "chain_announcements"
    / "ak_chain_announcements.csv"
)

BLOCKCHAIN_DIR = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
)

ADDRESS_TO_USERID_OUT = BLOCKCHAIN_DIR / "address_to_userid.csv"
OWNER_ADDRESS_OUT = BLOCKCHAIN_DIR / "owner_address.txt"
WALLET_MAPPING_REPORT_OUT = (
    ANNOUNCEMENT_ROOT
    / "outputs"
    / "logs"
    / "wallet_mapping_report.txt"
)


# =====================================================================================
# Cell 1 — Helpers
# =====================================================================================

def ensure_file_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def ensure_dirs() -> None:
    BLOCKCHAIN_DIR.mkdir(parents=True, exist_ok=True)
    WALLET_MAPPING_REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)


def connect_ganache(rpc_url: str) -> Web3:
    w3 = Web3(Web3.HTTPProvider(rpc_url))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {rpc_url}. "
            f"Make sure Ganache is running."
        )

    return w3


# =====================================================================================
# Cell 2 — Build Wallet Mapping
# =====================================================================================

def build_address_to_userid() -> pd.DataFrame:
    ensure_dirs()

    ensure_file_exists(COMMUNITY_MEMBERS_CSV, "community_members.csv")
    ensure_file_exists(AK_ANNOUNCEMENTS_CSV, "ak_chain_announcements.csv")

    df_members = pd.read_csv(COMMUNITY_MEMBERS_CSV)
    df_ak = pd.read_csv(AK_ANNOUNCEMENTS_CSV)

    required_member_cols = {"user_id", "community_id"}
    missing_members = required_member_cols - set(df_members.columns)
    if missing_members:
        raise ValueError(
            f"[ERROR] community_members.csv missing columns: {missing_members}"
        )

    required_ak_cols = {"community_id", "A_k"}
    missing_ak = required_ak_cols - set(df_ak.columns)
    if missing_ak:
        raise ValueError(
            f"[ERROR] ak_chain_announcements.csv missing columns: {missing_ak}"
        )

    df_members["user_id"] = df_members["user_id"].astype(int)
    df_members["community_id"] = df_members["community_id"].astype(int)

    df_ak["community_id"] = df_ak["community_id"].astype(int)
    df_ak["A_k"] = df_ak["A_k"].astype(int)

    member_user_ids = set(df_members["user_id"].dropna().astype(int).tolist())
    ak_user_ids = set(df_ak["A_k"].dropna().astype(int).tolist())

    all_required_users = sorted(member_user_ids | ak_user_ids)

    ak_not_in_membership = sorted(ak_user_ids - member_user_ids)

    w3 = connect_ganache(RPC_URL)

    accounts = [Web3.to_checksum_address(a) for a in w3.eth.accounts]

    if len(accounts) < 2:
        raise RuntimeError(
            "[ERROR] Ganache must have at least 2 accounts: "
            "one owner and at least one user account."
        )

    owner_address = accounts[0]
    available_user_accounts = accounts[1:]

    if len(available_user_accounts) < len(all_required_users):
        raise RuntimeError(
            "[ERROR] Not enough Ganache accounts.\n"
            f"Required user accounts: {len(all_required_users)}\n"
            f"Available user accounts: {len(available_user_accounts)}\n"
            f"Total Ganache accounts needed at least: {len(all_required_users) + 1}\n"
            "Restart Ganache with more accounts, for example:\n"
            "ganache --wallet.totalAccounts 750 --chain.chainId 1337 --server.port 7545"
        )

    selected_user_accounts = available_user_accounts[:len(all_required_users)]

    rows = []

    for user_id, address in zip(all_required_users, selected_user_accounts):
        rows.append({
            "user_id": int(user_id),
            "address": address,
            "is_community_member": bool(user_id in member_user_ids),
            "is_Ak": bool(user_id in ak_user_ids),
        })

    df_out = pd.DataFrame(rows).sort_values("user_id").reset_index(drop=True)

    df_out.to_csv(ADDRESS_TO_USERID_OUT, index=False, encoding="utf-8-sig")

    OWNER_ADDRESS_OUT.write_text(owner_address, encoding="utf-8")

    lines = []
    lines.append("=" * 90)
    lines.append("Ganache Wallet Mapping Report")
    lines.append("=" * 90)
    lines.append("")
    lines.append(f"RPC_URL                         : {RPC_URL}")
    lines.append(f"Owner address                   : {owner_address}")
    lines.append(f"Total Ganache accounts           : {len(accounts)}")
    lines.append(f"Community member users           : {len(member_user_ids)}")
    lines.append(f"Unique A_k users                 : {len(ak_user_ids)}")
    lines.append(f"Total mapped users               : {len(all_required_users)}")
    lines.append(f"A_k users not in community file  : {len(ak_not_in_membership)}")
    lines.append("")
    lines.append("A_k users not found in community_members.csv:")
    if ak_not_in_membership:
        for u in ak_not_in_membership:
            lines.append(f"- {u}")
    else:
        lines.append("None")
    lines.append("")
    lines.append(f"ADDRESS_TO_USERID_OUT            : {ADDRESS_TO_USERID_OUT}")
    lines.append(f"OWNER_ADDRESS_OUT                : {OWNER_ADDRESS_OUT}")
    lines.append("")

    WALLET_MAPPING_REPORT_OUT.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )

    print("=" * 90)
    print("[DONE] address_to_userid.csv created successfully")
    print("=" * 90)
    print("[PATH]", ADDRESS_TO_USERID_OUT)
    print("[PATH]", OWNER_ADDRESS_OUT)
    print("[PATH]", WALLET_MAPPING_REPORT_OUT)
    print("")
    print("[SUMMARY]")
    print("Owner address                 :", owner_address)
    print("Total Ganache accounts         :", len(accounts))
    print("Community member users         :", len(member_user_ids))
    print("Unique A_k users               :", len(ak_user_ids))
    print("Total mapped users             :", len(all_required_users))
    print("A_k not in community_members   :", ak_not_in_membership)
    print("")

    print(df_out.head(10).to_string(index=False))

    return df_out


# =====================================================================================
# Cell 3 — Run
# =====================================================================================

df_address_map = build_address_to_userid()

[DONE] address_to_userid.csv created successfully
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\data\blockchain\address_to_userid.csv
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\data\blockchain\owner_address.txt
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\outputs\logs\wallet_mapping_report.txt

[SUMMARY]
Owner address                 : 0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D
Total Ganache accounts         : 750
Community member users         : 693
Unique A_k users               : 5
Total mapped users             : 697
A_k not in community_members   : [28, 30, 289, 460]

 user_id                                    address  is_community_member  is_Ak
       1 0x7238AEDdBeB5A9a252ea8858A20EceDDD804541C                 True  False
       5 0xc3348b298A12cb52079E8923a2f50653BD89Cb24                 True  False
       9 0xaDcA3ACA212B015860781910F8F1EE30A0649c74                 True  False
      10 0x8945

### الفحص الاول
### هذا الكود للفحص هل الملفات التي يحتاجها العقد الذكي موجودة ام لا


In [3]:
from pathlib import Path
import pandas as pd

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

community_members = ANNOUNCEMENT_ROOT / "data" / "communities" / "community_members.csv"
ak_announcements = ANNOUNCEMENT_ROOT / "data" / "chain_announcements" / "ak_chain_announcements.csv"
address_map = ANNOUNCEMENT_ROOT / "data" / "blockchain" / "address_to_userid.csv"
owner_file = ANNOUNCEMENT_ROOT / "data" / "blockchain" / "owner_address.txt"

print("community_members exists:", community_members.exists())
print("ak_chain_announcements exists:", ak_announcements.exists())
print("address_to_userid exists:", address_map.exists())
print("owner_address exists:", owner_file.exists())

df_ak = pd.read_csv(ak_announcements)
df_addr = pd.read_csv(address_map)

print("\nColumns in ak_chain_announcements.csv:")
print(df_ak.columns.tolist())

print("\nOwner:")
print(owner_file.read_text(encoding="utf-8").strip())

print("\nA_k announcements:")
print(df_ak[["community_id", "A_k", "required_confirmations"]].to_string(index=False))

print("\nAddress mapping summary:")
print("Mapped users:", len(df_addr))
print("Community members:", df_addr["is_community_member"].sum())
print("A_k users:", df_addr["is_Ak"].sum())

assert "snapshot_id" not in df_ak.columns, "snapshot_id still exists. Re-run the first code after removing snapshot."

community_members exists: True
ak_chain_announcements exists: True
address_to_userid exists: True
owner_address exists: True

Columns in ak_chain_announcements.csv:
['community_id', 'A_k', 'result_hash', 'required_confirmations', 'community_size_used_on_chain', 'interest_size_from_summary', 'interest_size_from_membership_file', 'nominating_voters', 'abstain_count', 'unique_nominees', 'decision_basis', 'leader_in_I', 'json_file']

Owner:
0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D

A_k announcements:
 community_id  A_k  required_confirmations
            8  378                      53
           14  289                      69
           84   28                      59
           96   30                      69
          101   28                      41
          287  460                      60

Address mapping summary:
Mapped users: 697
Community members: 693
A_k users: 5


### الفحص الثاني

 فقط للتأكد من ان العقد في مساره الصحيح 
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\contracts\AuthorizedNodeElection.sol
حتى تتأكد من وجود العقد في مساره الصحيح، شغّل هذه الخلية في فقط للفحص###
JupyterLab:

In [4]:
from pathlib import Path

contract_path = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\contracts\AuthorizedNodeElection.sol"
)

print("Contract exists:", contract_path.exists())
print("Contract path:", contract_path)

if contract_path.exists():
    print("File size:", contract_path.stat().st_size, "bytes")

Contract exists: True
Contract path: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\contracts\AuthorizedNodeElection.sol
File size: 20827 bytes


### إضافة قياسات أداء Blockchain — معتمدة لهذه الإعادة

هذه الخلية لا تغيّر منطق النظام أو العقد الذكي أو اختيار $A_k$. وظيفتها فقط إنشاء سجل مركزي لكل معاملة On-chain لقياس Gas، زمن التنفيذ، الكلفة، والبيانات اللازمة لتحليل قابلية التوسع والـThroughput.


In [5]:
# -*- coding: utf-8 -*-
# =====================================================================================
# PERFORMANCE INSTRUMENTATION — Douban Movie
# قياسات إضافية فقط؛ لا تغيير في منطق العقد أو آلية الاختيار/التأكيد
# =====================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import time
import pandas as pd
from web3 import Web3

DATASET_NAME = "Douban Movie"
ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)
RPC_URL = "http://127.0.0.1:7545"

PERFORMANCE_ROOT = ANNOUNCEMENT_ROOT / "outputs" / "performance"
PERFORMANCE_ROOT.mkdir(parents=True, exist_ok=True)

# إذا أُعيد تنفيذ هذه الخلية داخل نفس الـKernel لا ننشئ Run جديدًا بالخطأ.
if "PERF_RUN_ID" not in globals():
    PERF_RUN_ID = datetime.now().strftime("PERF_%Y%m%d_%H%M%S")

PERF_RUN_DIR = PERFORMANCE_ROOT / PERF_RUN_ID
PERF_RUN_DIR.mkdir(parents=True, exist_ok=True)

PERF_TX_CSV = PERF_RUN_DIR / "blockchain_performance_transactions.csv"
PERF_STAGE_SUMMARY_CSV = PERF_RUN_DIR / "blockchain_performance_by_stage.csv"
PERF_COMMUNITY_SUMMARY_CSV = PERF_RUN_DIR / "blockchain_performance_by_community.csv"
PERF_RUN_META_JSON = PERF_RUN_DIR / "run_metadata.json"
PERF_CURRENT_RUN_FILE = PERFORMANCE_ROOT / "current_performance_run.txt"

PERF_CURRENT_RUN_FILE.write_text(PERF_RUN_ID, encoding="utf-8")

PERF_COLUMNS = [
    "dataset", "run_id", "chain_id", "stage", "transaction_type",
    "community_id", "community_size", "required_confirmations",
    "batch_number", "batch_size", "confirmation_count_before",
    "confirmation_count_after", "is_finalizing", "stage_elapsed_sec",
    "sender_address", "contract_address", "tx_hash", "receipt_status",
    "block_number", "block_timestamp", "block_gas_used", "block_gas_limit",
    "tx_gas_limit", "gas_used", "effective_gas_price_wei",
    "effective_gas_price_gwei", "cost_wei", "cost_eth", "latency_sec",
    "submitted_at_utc", "receipt_at_utc", "note",
]


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def measure_transaction(w3: Web3, tx_sender, gas_limit: int) -> tuple:
    """Send one transaction, wait for its receipt, and return receipt + measured metrics."""
    submitted_at_utc = utc_now_iso()
    t0 = time.perf_counter()
    tx_hash = tx_sender()
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    t1 = time.perf_counter()
    receipt_at_utc = utc_now_iso()

    tx = w3.eth.get_transaction(tx_hash)
    block = w3.eth.get_block(receipt.blockNumber)
    gas_used = int(receipt.gasUsed)

    effective_gas_price = receipt.get("effectiveGasPrice", None)
    if effective_gas_price is None:
        effective_gas_price = tx.get("gasPrice", 0)
    effective_gas_price = int(effective_gas_price or 0)
    cost_wei = int(gas_used * effective_gas_price)

    metrics = {
        "chain_id": int(w3.eth.chain_id),
        "tx_hash": Web3.to_hex(tx_hash),
        "receipt_status": int(receipt.status),
        "block_number": int(receipt.blockNumber),
        "block_timestamp": int(block["timestamp"]),
        "block_gas_used": int(block.get("gasUsed", 0)),
        "block_gas_limit": int(block.get("gasLimit", 0)),
        "tx_gas_limit": int(gas_limit),
        "gas_used": gas_used,
        "effective_gas_price_wei": effective_gas_price,
        "effective_gas_price_gwei": effective_gas_price / 1e9,
        "cost_wei": cost_wei,
        "cost_eth": cost_wei / 1e18,
        "latency_sec": float(t1 - t0),
        "submitted_at_utc": submitted_at_utc,
        "receipt_at_utc": receipt_at_utc,
    }
    return receipt, metrics


def append_performance_row(**values) -> None:
    """Append one transaction row immediately, so partial measurements are not lost."""
    row = {col: None for col in PERF_COLUMNS}
    row.update({"dataset": DATASET_NAME, "run_id": PERF_RUN_ID})
    row.update(values)
    pd.DataFrame([row], columns=PERF_COLUMNS).to_csv(
        PERF_TX_CSV,
        mode="a",
        header=not PERF_TX_CSV.exists(),
        index=False,
        encoding="utf-8-sig",
    )


def update_run_metadata(**updates) -> None:
    meta = {}
    if PERF_RUN_META_JSON.exists():
        try:
            meta = json.loads(PERF_RUN_META_JSON.read_text(encoding="utf-8"))
        except Exception:
            meta = {}
    meta.update({
        "dataset": DATASET_NAME,
        "run_id": PERF_RUN_ID,
        "rpc_url": RPC_URL,
        "measurement_method": "Python Web3 submit-to-receipt timing using time.perf_counter; gas from transaction receipt.gasUsed",
        "measurement_scope": "Contract deployment, community-member loading, authorized-node announcement, member confirmations, and finalization-triggering confirmation",
        "note": "Local Ganache measurements; monetary cost uses Ganache effective gas price and is not a public-network fee estimate.",
    })
    meta.update(updates)
    PERF_RUN_META_JSON.write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
    )


w3_perf_check = Web3(Web3.HTTPProvider(RPC_URL))
update_run_metadata(
    started_at_utc=utc_now_iso(),
    connected=bool(w3_perf_check.is_connected()),
    chain_id=int(w3_perf_check.eth.chain_id) if w3_perf_check.is_connected() else None,
    starting_block=int(w3_perf_check.eth.block_number) if w3_perf_check.is_connected() else None,
    ganache_accounts=len(w3_perf_check.eth.accounts) if w3_perf_check.is_connected() else None,
)

print("[PERFORMANCE] Run ID:", PERF_RUN_ID)
print("[PERFORMANCE] Transaction log:", PERF_TX_CSV)
print("[PERFORMANCE] Metadata:", PERF_RUN_META_JSON)


[PERFORMANCE] Run ID: PERF_20260901_003219
[PERFORMANCE] Transaction log: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\outputs\performance\PERF_20260901_003219\blockchain_performance_transactions.csv
[PERFORMANCE] Metadata: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\outputs\performance\PERF_20260901_003219\run_metadata.json


### الخطوة 3
نشر العقد بواسطة بايثون
هذا هو كود نشر العقد الذكي على 
Ganache بواسطة 
Python، 
بدل استخدام زر 
Deploy في Remix.

In [6]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Deploy AuthorizedNodeElection.sol to Ganache using Python Web3
# =====================================================================================

from pathlib import Path
import json

from solcx import compile_source, install_solc
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

CONTRACT_FILE = ANNOUNCEMENT_ROOT / "contracts" / "AuthorizedNodeElection.sol"
BUILD_DIR = ANNOUNCEMENT_ROOT / "contracts" / "build"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

ABI_OUT = BUILD_DIR / "AuthorizedNodeElection.abi.json"
ADDRESS_OUT = BUILD_DIR / "AuthorizedNodeElection.address.txt"

OWNER_ADDRESS_FILE = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
    / "owner_address.txt"
)

RPC_URL = "http://127.0.0.1:7545"
CHAIN_ID = 1337

# مدة التأكيد بالثواني
# 3600 = ساعة واحدة
CONFIRMATION_WINDOW_SECONDS = 3600


# =====================================================================================
# Checks
# =====================================================================================

if not CONTRACT_FILE.exists():
    raise FileNotFoundError(f"Contract file not found: {CONTRACT_FILE}")

if not OWNER_ADDRESS_FILE.exists():
    raise FileNotFoundError(f"Owner address file not found: {OWNER_ADDRESS_FILE}")


# =====================================================================================
# Connect to Ganache
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))


# =====================================================================================
# Owner account
# =====================================================================================

owner_address = OWNER_ADDRESS_FILE.read_text(encoding="utf-8").strip()
owner_address = Web3.to_checksum_address(owner_address)

print("[INFO] Owner from file:", owner_address)
print("[INFO] Ganache account[0]:", w3.eth.accounts[0])

if owner_address.lower() != w3.eth.accounts[0].lower():
    raise RuntimeError(
        "Owner address in owner_address.txt does not match Ganache account[0]. "
        "Re-run address_to_userid mapping with the current Ganache session."
    )


# =====================================================================================
# Compile Solidity
# =====================================================================================

print("[INFO] Installing/using solc 0.8.20...")
install_solc("0.8.20")

source_code = CONTRACT_FILE.read_text(encoding="utf-8")

compiled = compile_source(
    source_code,
    output_values=["abi", "bin"],
    solc_version="0.8.20",
    optimize=True,
)

contract_id, contract_interface = compiled.popitem()

abi = contract_interface["abi"]
bytecode = contract_interface["bin"]

print("[OK] Contract compiled")
print("[INFO] Contract ID:", contract_id)


# =====================================================================================
# Deploy
# =====================================================================================

Contract = w3.eth.contract(abi=abi, bytecode=bytecode)

DEPLOY_GAS_LIMIT = 7_000_000

receipt, perf = measure_transaction(
    w3,
    lambda: Contract.constructor(CONFIRMATION_WINDOW_SECONDS).transact({
        "from": owner_address,
        "gas": DEPLOY_GAS_LIMIT,
    }),
    gas_limit=DEPLOY_GAS_LIMIT,
)

contract_address = receipt.contractAddress

append_performance_row(
    stage="deployment",
    transaction_type="Contract deployment",
    sender_address=owner_address,
    contract_address=contract_address,
    note="One-time deployment cost",
    **perf,
)

update_run_metadata(
    contract_address=str(contract_address),
    confirmation_window_seconds=int(CONFIRMATION_WINDOW_SECONDS),
    solidity_compiler="0.8.20",
    compiler_optimization=True,
)

print("[OK] Contract deployed")
print("[INFO] Tx hash:", perf["tx_hash"])
print("[INFO] Contract address:", contract_address)
print("[PERFORMANCE] gasUsed:", perf["gas_used"])
print("[PERFORMANCE] latency_sec:", f'{perf["latency_sec"]:.6f}')


# =====================================================================================
# Save ABI and address
# =====================================================================================

ABI_OUT.write_text(
    json.dumps(abi, indent=2),
    encoding="utf-8"
)

ADDRESS_OUT.write_text(
    contract_address,
    encoding="utf-8"
)

print("[OK] ABI saved to:", ABI_OUT)
print("[OK] Address saved to:", ADDRESS_OUT)


# =====================================================================================
# Verify deployment
# =====================================================================================

contract = w3.eth.contract(
    address=Web3.to_checksum_address(contract_address),
    abi=abi
)

print("\n[VERIFY]")
print("owner on-chain                 :", contract.functions.owner().call())
print("confirmationWindowSeconds      :", contract.functions.confirmationWindowSeconds().call())
print("communitySize(8) before loading:", contract.functions.communitySize(8).call())

[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 750
[INFO] Owner from file: 0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D
[INFO] Ganache account[0]: 0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D
[INFO] Installing/using solc 0.8.20...
[OK] Contract compiled
[INFO] Contract ID: <stdin>:AuthorizedNodeElection
[OK] Contract deployed
[INFO] Tx hash: 0x5a71a1268c281c6f0623b3431f8a5a7d6775d51186e7525a0e7dd2ab7926a416
[INFO] Contract address: 0x27fFA41Dd35f806E2e8b320c1f074B7E33d8B902
[PERFORMANCE] gasUsed: 1271695
[PERFORMANCE] latency_sec: 0.073753
[OK] ABI saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\contracts\build\AuthorizedNodeElection.abi.json
[OK] Address saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\contracts\build\AuthorizedNodeElection.address.txt

[VERIFY]
owner on-chain                 : 0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D
confirmationWindowSeconds      : 3600
communitySize(8) before loading: 0


### الخطوة 4
3. الخطوة التالية الآن
الخطوة التالية هي:
تحميل أعضاء المجتمعات إلى العقد الذكي
يعني سنأخذ أعضاء كل مجتمع من:
community_members.csv

ونحوّلهم إلى عناوين 
wallet باستخدام:
address_to_userid.csv

ثم نرسلهم إلى دالة العقد:
loadCommunityMembers(communityId, members)

بعدها يصبح العقد يعرف من هم أعضاء كل مجتمع، وبذلك يسمح لهم لاحقًا بإرسال:
announceAk(...)

In [7]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Load Community Members into AuthorizedNodeElection Smart Contract
# =====================================================================================
#
# الهدف:
# تحميل أعضاء كل مجتمع إلى العقد الذكي حتى يعرف العقد من يحق له الإعلان والتأكيد.
#
# يعتمد على:
# 1) community_members.csv
# 2) address_to_userid.csv
# 3) AuthorizedNodeElection.abi.json
# 4) AuthorizedNodeElection.address.txt
#
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

COMMUNITY_MEMBERS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "communities"
    / "community_members.csv"
)

ADDRESS_TO_USERID_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
    / "address_to_userid.csv"
)

OWNER_ADDRESS_FILE = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
    / "owner_address.txt"
)

ABI_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.abi.json"
)

CONTRACT_ADDRESS_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.address.txt"
)

LOAD_REPORT_OUT = (
    ANNOUNCEMENT_ROOT
    / "outputs"
    / "logs"
    / "load_members_report.csv"
)

LOAD_REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# Settings
# =====================================================================================

BATCH_SIZE = 50
TX_GAS = 8_000_000


# =====================================================================================
# Checks
# =====================================================================================

required_files = [
    COMMUNITY_MEMBERS_CSV,
    ADDRESS_TO_USERID_CSV,
    OWNER_ADDRESS_FILE,
    ABI_PATH,
    CONTRACT_ADDRESS_PATH,
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")


# =====================================================================================
# Connect to Ganache and Contract
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))


owner_address = OWNER_ADDRESS_FILE.read_text(encoding="utf-8").strip()
owner_address = Web3.to_checksum_address(owner_address)

contract_address = CONTRACT_ADDRESS_PATH.read_text(encoding="utf-8").strip()
contract_address = Web3.to_checksum_address(contract_address)

abi = json.loads(ABI_PATH.read_text(encoding="utf-8"))

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

onchain_owner = contract.functions.owner().call()

print("[INFO] Owner from file:", owner_address)
print("[INFO] Owner on-chain :", onchain_owner)
print("[INFO] Contract address:", contract_address)

if owner_address.lower() != onchain_owner.lower():
    raise RuntimeError(
        "Owner mismatch. The owner_address.txt does not match contract owner."
    )


# =====================================================================================
# Load CSV files
# =====================================================================================

df_members = pd.read_csv(COMMUNITY_MEMBERS_CSV)
df_addr = pd.read_csv(ADDRESS_TO_USERID_CSV)

df_members["user_id"] = df_members["user_id"].astype(int)
df_members["community_id"] = df_members["community_id"].astype(int)

df_addr["user_id"] = df_addr["user_id"].astype(int)
df_addr["address"] = df_addr["address"].astype(str)

user_to_address = dict(
    zip(
        df_addr["user_id"].astype(int),
        df_addr["address"].astype(str)
    )
)


# =====================================================================================
# Load members community by community
# =====================================================================================

report_rows = []

community_ids = sorted(df_members["community_id"].unique().tolist())

print("\n[START] Loading community members to contract...")
print("[INFO] Communities:", community_ids)

for community_id in community_ids:
    user_ids = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    missing_users = [u for u in user_ids if u not in user_to_address]

    if missing_users:
        raise RuntimeError(
            f"Missing wallet addresses for community {community_id}: {missing_users[:10]}"
        )

    addresses = [
        Web3.to_checksum_address(user_to_address[u])
        for u in user_ids
    ]

    before_size = contract.functions.communitySize(int(community_id)).call()

    print("\n" + "=" * 90)
    print(f"[COMMUNITY] {community_id}")
    print("[INFO] Users to load:", len(user_ids))
    print("[INFO] Current on-chain size before loading:", before_size)

    tx_count = 0
    load_stage_t0 = time.perf_counter()
    expected_required = (len(user_ids) // 2) + 1

    for i in range(0, len(addresses), BATCH_SIZE):
        batch = addresses[i:i + BATCH_SIZE]
        batch_number = tx_count + 1

        receipt, perf = measure_transaction(
            w3,
            lambda batch=batch: contract.functions.loadCommunityMembers(
                int(community_id),
                batch
            ).transact({
                "from": owner_address,
                "gas": TX_GAS
            }),
            gas_limit=TX_GAS,
        )

        tx_count += 1
        stage_elapsed = time.perf_counter() - load_stage_t0

        append_performance_row(
            stage="member_loading",
            transaction_type="Community-member loading",
            community_id=int(community_id),
            community_size=int(len(user_ids)),
            required_confirmations=int(expected_required),
            batch_number=int(batch_number),
            batch_size=int(len(batch)),
            stage_elapsed_sec=float(stage_elapsed),
            sender_address=owner_address,
            contract_address=contract_address,
            note="Setup cost; kept separate from authorization/finalization cost",
            **perf,
        )

        print(
            f"[OK] Batch {tx_count} loaded | "
            f"size={len(batch)} | tx={perf['tx_hash']} | "
            f"gasUsed={perf['gas_used']} | latency={perf['latency_sec']:.6f}s"
        )

    after_size = contract.functions.communitySize(int(community_id)).call()
    required = contract.functions.requiredConfirmations(int(community_id)).call()

    print("[INFO] On-chain size after loading:", after_size)
    print("[INFO] Required confirmations:", required)

    report_rows.append({
        "community_id": int(community_id),
        "expected_members": int(len(user_ids)),
        "onchain_size_before": int(before_size),
        "onchain_size_after": int(after_size),
        "required_confirmations": int(required),
        "transactions_sent": int(tx_count),
    })


# =====================================================================================
# Save report
# =====================================================================================

df_report = pd.DataFrame(report_rows)
df_report.to_csv(LOAD_REPORT_OUT, index=False, encoding="utf-8-sig")

print("\n" + "=" * 90)
print("[DONE] Community members loaded successfully.")
print("[PATH] Load report:", LOAD_REPORT_OUT)
print("\n[SUMMARY]")
print(df_report.to_string(index=False))

[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 750
[INFO] Owner from file: 0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D
[INFO] Owner on-chain : 0x297A2E1d3a11579E25C1A4c9AD1d83DB3cfe992D
[INFO] Contract address: 0x27fFA41Dd35f806E2e8b320c1f074B7E33d8B902

[START] Loading community members to contract...
[INFO] Communities: [8, 14, 84, 96, 101, 287]

[COMMUNITY] 8
[INFO] Users to load: 104
[INFO] Current on-chain size before loading: 0
[OK] Batch 1 loaded | size=50 | tx=0x299c07aa9602131da9e3b53f0100582232be72257dad6ec65397d15175dcfaa8 | gasUsed=1235844 | latency=0.313212s
[OK] Batch 2 loaded | size=50 | tx=0xb859e0ef25bf6494fe7862189d8a0c857d3f37ddd57d68f8932bf6b3116d7a5f | gasUsed=1218732 | latency=0.274384s
[OK] Batch 3 loaded | size=4 | tx=0xa53db63a281c4bac547a78a086475de1f2f6cfa97f9823e1087956e93b6e67fd | gasUsed=125924 | latency=0.039888s
[INFO] On-chain size after loading: 104
[INFO] Required confirmations: 53

[COMMUNITY] 14
[INFO] Users to load: 137
[INFO] Curr

### الخطوة 6 
4. كود إعلان
A_k لكل مجتمع
يعني الاعلان لجميع المجتمعات  و ليس لمجتمع واحد فقط 
و حيث ستقوم احد النودات التي تنتمي الى المجتمع بالاعلان عن النود الترستد و حسب التسلسل التالي
1. قراءة community_id و A_k و result_hash.
2. اختيار عضو من نفس المجتمع كـ announcer.
3. التأكد أن هذا العنوان عضو داخل العقد.
4. إرسال announceAk.
5. يصبح confirmCount = 1.
6. يبدأ وقت التأكيد deadline لمدة 3600 ثانية.

******************************
هنا يتم إعلان الـ 
Trusted Authorized Node A_k
 داخل العقد الذكي، لكن انتبه للفرق المهم:

announceAk = إعلان القائد 
confirmAk  = تأكيد الإعلان
finalize   = تثبيت القائد رسميًا بعد الأغلبية

العمل في announceAk = إعلان القائد 

In [8]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Announce A_k for Each Community on AuthorizedNodeElection Smart Contract
# =====================================================================================
#
# الهدف:
# إرسال إعلان A_k لكل مجتمع إلى العقد الذكي.
#
# يعتمد على:
# 1) ak_chain_announcements.csv
# 2) community_members.csv
# 3) address_to_userid.csv
# 4) AuthorizedNodeElection.abi.json
# 5) AuthorizedNodeElection.address.txt
#
# ملاحظة:
# المعلن يجب أن يكون عضوًا في المجتمع.
# لذلك نختار أول عضو من كل مجتمع كمعلن.
#
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

AK_ANNOUNCEMENTS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "chain_announcements"
    / "ak_chain_announcements.csv"
)

COMMUNITY_MEMBERS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "communities"
    / "community_members.csv"
)

ADDRESS_TO_USERID_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
    / "address_to_userid.csv"
)

ABI_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.abi.json"
)

CONTRACT_ADDRESS_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.address.txt"
)

ANNOUNCE_REPORT_OUT = (
    ANNOUNCEMENT_ROOT
    / "outputs"
    / "logs"
    / "announce_ak_report.csv"
)

ANNOUNCE_REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# Settings
# =====================================================================================

TX_GAS = 700_000


# =====================================================================================
# Checks
# =====================================================================================

required_files = [
    AK_ANNOUNCEMENTS_CSV,
    COMMUNITY_MEMBERS_CSV,
    ADDRESS_TO_USERID_CSV,
    ABI_PATH,
    CONTRACT_ADDRESS_PATH,
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")


# =====================================================================================
# Connect to Ganache and Contract
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))

abi = json.loads(ABI_PATH.read_text(encoding="utf-8"))
contract_address = CONTRACT_ADDRESS_PATH.read_text(encoding="utf-8").strip()
contract_address = Web3.to_checksum_address(contract_address)

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

print("[INFO] Contract address:", contract_address)
print("[INFO] confirmationWindowSeconds:", contract.functions.confirmationWindowSeconds().call())


# =====================================================================================
# Load CSV files
# =====================================================================================

df_ak = pd.read_csv(AK_ANNOUNCEMENTS_CSV)
df_members = pd.read_csv(COMMUNITY_MEMBERS_CSV)
df_addr = pd.read_csv(ADDRESS_TO_USERID_CSV)

df_ak["community_id"] = df_ak["community_id"].astype(int)
df_ak["A_k"] = df_ak["A_k"].astype(int)

df_members["community_id"] = df_members["community_id"].astype(int)
df_members["user_id"] = df_members["user_id"].astype(int)

df_addr["user_id"] = df_addr["user_id"].astype(int)
df_addr["address"] = df_addr["address"].astype(str)

user_to_address = dict(
    zip(
        df_addr["user_id"].astype(int),
        df_addr["address"].astype(str)
    )
)


# =====================================================================================
# Helper functions
# =====================================================================================

def result_hash_to_bytes32(result_hash_hex: str) -> bytes:
    """
    تحويل result_hash من hex string إلى bytes32 مناسب للعقد.
    """
    h = str(result_hash_hex).strip()

    if h.startswith("0x"):
        h = h[2:]

    if len(h) != 64:
        raise ValueError(f"Invalid result_hash length. Expected 64 hex chars, got {len(h)}")

    return bytes.fromhex(h)


def select_announcer_for_community(community_id: int) -> tuple[int, str]:
    """
    اختيار أول عضو في المجتمع كمعلن.
    """
    user_ids = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    if not user_ids:
        raise RuntimeError(f"No members found for community {community_id}")

    announcer_user_id = int(user_ids[0])

    if announcer_user_id not in user_to_address:
        raise RuntimeError(
            f"Missing wallet address for announcer user_id={announcer_user_id}"
        )

    announcer_address = Web3.to_checksum_address(user_to_address[announcer_user_id])

    return announcer_user_id, announcer_address


# =====================================================================================
# Announce A_k community by community
# =====================================================================================

report_rows = []

print("\n[START] Announcing A_k for communities...")

for _, row in df_ak.sort_values("community_id").iterrows():
    community_id = int(row["community_id"])
    Ak = int(row["A_k"])
    result_hash_hex = str(row["result_hash"])
    result_hash_bytes = result_hash_to_bytes32(result_hash_hex)

    announcer_user_id, announcer_address = select_announcer_for_community(community_id)

    print("\n" + "=" * 90)
    print(f"[COMMUNITY] {community_id}")
    print("[INFO] A_k:", Ak)
    print("[INFO] Announcer user_id:", announcer_user_id)
    print("[INFO] Announcer address:", announcer_address)

    # تأكد أن المعلن عضو في العقد
    is_member = contract.functions.isMember(
        int(community_id),
        announcer_address
    ).call()

    print("[INFO] Announcer is member on-chain:", is_member)

    if not is_member:
        raise RuntimeError(
            f"Selected announcer is not a member on-chain. "
            f"community={community_id}, user_id={announcer_user_id}"
        )

    # قراءة حالة الإعلان قبل الإرسال
    before = contract.functions.getAnnouncement(int(community_id)).call()
    before_Ak = before[0]
    before_confirm_count = before[2]
    before_exists = before[3]
    before_finalized = before[4]

    if before_finalized:
        print("[SKIP] A_k already finalized for this community.")
        status = "already_finalized"
        tx_hash_hex = None

    elif before_exists:
        print("[SKIP] Pending announcement already exists.")
        status = "already_pending"
        tx_hash_hex = None

    else:
        info_before_announce = contract.functions.getCommunityConfirmationInfo(
            int(community_id)
        ).call()
        community_size_before_announce = int(info_before_announce[0])
        required_before_announce = int(info_before_announce[1])
        confirm_before_announce = int(info_before_announce[2])

        receipt, perf = measure_transaction(
            w3,
            lambda: contract.functions.announceAk(
                int(community_id),
                int(Ak),
                result_hash_bytes
            ).transact({
                "from": announcer_address,
                "gas": TX_GAS
            }),
            gas_limit=TX_GAS,
        )
        tx_hash_hex = perf["tx_hash"]

        after_for_perf = contract.functions.getAnnouncement(int(community_id)).call()
        confirm_after_announce = int(after_for_perf[2])
        finalized_after_announce = bool(after_for_perf[4])

        append_performance_row(
            stage="announcement",
            transaction_type="Authorized-node announcement",
            community_id=int(community_id),
            community_size=int(community_size_before_announce),
            required_confirmations=int(required_before_announce),
            confirmation_count_before=int(confirm_before_announce),
            confirmation_count_after=int(confirm_after_announce),
            is_finalizing=bool(finalized_after_announce),
            stage_elapsed_sec=float(perf["latency_sec"]),
            sender_address=announcer_address,
            contract_address=contract_address,
            note="Announcement is counted as the first confirmation by the implemented mechanism",
            **perf,
        )

        print("[OK] A_k announcement sent.")
        print("[INFO] Tx:", tx_hash_hex)
        print("[PERFORMANCE] gasUsed:", perf["gas_used"])
        print("[PERFORMANCE] latency_sec:", f'{perf["latency_sec"]:.6f}')

        status = "announced"

    # قراءة الحالة بعد الإعلان
    after = contract.functions.getAnnouncement(int(community_id)).call()
    timing = contract.functions.getAnnouncementTiming(int(community_id)).call()
    info = contract.functions.getCommunityConfirmationInfo(int(community_id)).call()

    after_Ak = int(after[0])
    after_result_hash = after[1].hex()
    after_confirm_count = int(after[2])
    after_exists = bool(after[3])
    after_finalized = bool(after[4])
    after_announcer = after[5]

    announced_at = int(timing[0])
    deadline = int(timing[1])
    expired = bool(timing[2])

    onchain_size = int(info[0])
    required_count = int(info[1])
    current_confirm_count = int(info[2])

    print("[STATE]")
    print("exists       :", after_exists)
    print("finalized    :", after_finalized)
    print("confirmCount :", after_confirm_count)
    print("required     :", required_count)
    print("deadline     :", deadline)
    print("expired      :", expired)

    report_rows.append({
        "community_id": community_id,
        "A_k_expected": Ak,
        "A_k_onchain": after_Ak,
        "result_hash_expected": result_hash_hex,
        "result_hash_onchain": after_result_hash,
        "announcer_user_id": announcer_user_id,
        "announcer_address": announcer_address,
        "status": status,
        "tx_hash": tx_hash_hex,
        "exists": after_exists,
        "finalized": after_finalized,
        "confirm_count": after_confirm_count,
        "required_confirmations": required_count,
        "community_size_onchain": onchain_size,
        "announced_at": announced_at,
        "deadline": deadline,
        "expired": expired,
    })


# =====================================================================================
# Save report
# =====================================================================================

df_report = pd.DataFrame(report_rows)
df_report.to_csv(ANNOUNCE_REPORT_OUT, index=False, encoding="utf-8-sig")

print("\n" + "=" * 90)
print("[DONE] A_k announcements completed.")
print("[PATH] Announce report:", ANNOUNCE_REPORT_OUT)
print("\n[SUMMARY]")
print(
    df_report[
        [
            "community_id",
            "A_k_expected",
            "status",
            "exists",
            "finalized",
            "confirm_count",
            "required_confirmations",
            "expired",
        ]
    ].to_string(index=False)
)

[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 750
[INFO] Contract address: 0x27fFA41Dd35f806E2e8b320c1f074B7E33d8B902
[INFO] confirmationWindowSeconds: 3600

[START] Announcing A_k for communities...

[COMMUNITY] 8
[INFO] A_k: 378
[INFO] Announcer user_id: 13
[INFO] Announcer address: 0x7F6740D24949E7E77EaB673Cf4A651F09bc7BfdD
[INFO] Announcer is member on-chain: True
[OK] A_k announcement sent.
[INFO] Tx: 0x9790630019e8cb35e35768ba8fa6e28c697ace4928f998ea6aafd9043e322cf1
[PERFORMANCE] gasUsed: 234331
[PERFORMANCE] latency_sec: 0.044045
[STATE]
exists       : True
finalized    : False
confirmCount : 1
required     : 53
deadline     : 1788215600
expired      : False

[COMMUNITY] 14
[INFO] A_k: 289
[INFO] Announcer user_id: 5
[INFO] Announcer address: 0xc3348b298A12cb52079E8923a2f50653BD89Cb24
[INFO] Announcer is member on-chain: True
[OK] A_k announcement sent.
[INFO] Tx: 0x24ff69e720c1e89c9737f51d486a07d041a49374db0ecbbe005a230d89ce7cb7
[PERFORMANCE] gasUsed: 234331


### الخطوة 7
2. ماذا سيعمل كود التأكيد؟
لكل مجتمع:

1. يقرأ A_k
 المعلن من العقد.
2. يقرأ عدد التأكيدات الحالي.
3. يقرأ العدد المطلوب requiredConfirmations.
4. يختار أعضاء من نفس المجتمع.
يرسل منهم معاملات التأكيدات 
5. يتوقف عندما يصل إلى العدد المطلوب.
6. يتأكد أن:
    finalized = True

ملاحظة مهمة قبل التشغيل
بما أن العقد عنده مهلة تأكيد:
confirmationWindowSeconds = 3600
فإذا مرّت ساعة من وقت الإعلان، فإن 
confirmAk سيفشل برسالة:
Confirmation deadline passed

In [9]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Confirm A_k Announcements Until Finalization
# =====================================================================================
#
# الهدف:
# إرسال confirmAk من أعضاء كل مجتمع حتى يصل الإعلان إلى الأغلبية البسيطة
# ويتم تثبيت A_k نهائيًا داخل العقد الذكي.
#
# يعتمد على:
# 1) community_members.csv
# 2) address_to_userid.csv
# 3) AuthorizedNodeElection.abi.json
# 4) AuthorizedNodeElection.address.txt
#
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

COMMUNITY_MEMBERS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "communities"
    / "community_members.csv"
)

ADDRESS_TO_USERID_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
    / "address_to_userid.csv"
)

ABI_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.abi.json"
)

CONTRACT_ADDRESS_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.address.txt"
)

CONFIRM_REPORT_OUT = (
    ANNOUNCEMENT_ROOT
    / "outputs"
    / "logs"
    / "confirm_ak_report.csv"
)

CONFIRM_REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# Settings
# =====================================================================================

TX_GAS = 300_000


# =====================================================================================
# Checks
# =====================================================================================

required_files = [
    COMMUNITY_MEMBERS_CSV,
    ADDRESS_TO_USERID_CSV,
    ABI_PATH,
    CONTRACT_ADDRESS_PATH,
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")


# =====================================================================================
# Connect to Ganache and Contract
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))

abi = json.loads(ABI_PATH.read_text(encoding="utf-8"))
contract_address = CONTRACT_ADDRESS_PATH.read_text(encoding="utf-8").strip()
contract_address = Web3.to_checksum_address(contract_address)

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

print("[INFO] Contract address:", contract_address)
print("[INFO] confirmationWindowSeconds:", contract.functions.confirmationWindowSeconds().call())


# =====================================================================================
# Load CSV files
# =====================================================================================

df_members = pd.read_csv(COMMUNITY_MEMBERS_CSV)
df_addr = pd.read_csv(ADDRESS_TO_USERID_CSV)

df_members["community_id"] = df_members["community_id"].astype(int)
df_members["user_id"] = df_members["user_id"].astype(int)

df_addr["user_id"] = df_addr["user_id"].astype(int)
df_addr["address"] = df_addr["address"].astype(str)

user_to_address = dict(
    zip(
        df_addr["user_id"].astype(int),
        df_addr["address"].astype(str)
    )
)


# =====================================================================================
# Helper
# =====================================================================================

def get_member_addresses_for_community(community_id: int) -> list[tuple[int, str]]:
    """
    يرجع قائمة أعضاء المجتمع مع عناوينهم:
    [(user_id, address), ...]
    """
    user_ids = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    out = []

    for u in user_ids:
        if u not in user_to_address:
            raise RuntimeError(
                f"Missing wallet address for user_id={u} in community={community_id}"
            )

        out.append((int(u), Web3.to_checksum_address(user_to_address[u])))

    return out


# =====================================================================================
# Confirm community by community
# =====================================================================================

report_rows = []

community_ids = sorted(df_members["community_id"].unique().tolist())

print("\n[START] Confirming A_k announcements...")

for community_id in community_ids:
    print("\n" + "=" * 90)
    print(f"[COMMUNITY] {community_id}")

    # Read current announcement
    ann = contract.functions.getAnnouncement(int(community_id)).call()
    timing = contract.functions.getAnnouncementTiming(int(community_id)).call()
    info = contract.functions.getCommunityConfirmationInfo(int(community_id)).call()

    Ak = int(ann[0])
    result_hash_onchain = ann[1].hex()
    confirm_count = int(ann[2])
    exists = bool(ann[3])
    finalized = bool(ann[4])
    announcer = ann[5]

    announced_at = int(timing[0])
    deadline = int(timing[1])
    expired = bool(timing[2])

    community_size = int(info[0])
    required_count = int(info[1])
    current_confirm_count = int(info[2])

    print("[STATE BEFORE]")
    print("exists       :", exists)
    print("finalized    :", finalized)
    print("A_k          :", Ak)
    print("confirmCount :", current_confirm_count)
    print("required     :", required_count)
    print("expired      :", expired)
    print("deadline     :", deadline)

    status = "not_processed"
    tx_sent = 0
    last_tx_hash = None

    if not exists:
        print("[SKIP] No pending announcement for this community.")
        status = "no_pending_announcement"

    elif finalized:
        print("[SKIP] Already finalized.")
        status = "already_finalized"

    elif expired:
        print("[SKIP] Announcement expired. Need expirePendingAnnouncement and re-announce.")
        status = "expired_before_confirmation"

    else:
        members = get_member_addresses_for_community(community_id)
        confirmation_stage_t0 = time.perf_counter()

        for user_id, addr in members:
            # إذا وصلنا للعدد المطلوب، توقف
            info_now = contract.functions.getCommunityConfirmationInfo(int(community_id)).call()
            current_confirm_count = int(info_now[2])

            ann_now = contract.functions.getAnnouncement(int(community_id)).call()
            finalized_now = bool(ann_now[4])

            if finalized_now:
                print("[OK] Finalized reached.")
                status = "finalized"
                break

            if current_confirm_count >= required_count:
                print("[OK] Required confirmation count reached.")
                status = "threshold_reached"
                break

            # لا نرسل تأكيدًا من نفس الشخص مرتين
            already_confirmed = contract.functions.hasAddressConfirmed(
                int(community_id),
                addr
            ).call()

            if already_confirmed:
                continue

            # تأكد أن العنوان عضو في المجتمع داخل العقد
            is_member = contract.functions.isMember(
                int(community_id),
                addr
            ).call()

            if not is_member:
                continue

            try:
                confirm_count_before_tx = int(current_confirm_count)

                receipt, perf = measure_transaction(
                    w3,
                    lambda: contract.functions.confirmAk(
                        int(community_id),
                        int(Ak)
                    ).transact({
                        "from": addr,
                        "gas": TX_GAS
                    }),
                    gas_limit=TX_GAS,
                )

                tx_sent += 1
                last_tx_hash = perf["tx_hash"]

                info_after_tx = contract.functions.getCommunityConfirmationInfo(
                    int(community_id)
                ).call()
                ann_after_tx = contract.functions.getAnnouncement(
                    int(community_id)
                ).call()

                confirm_count_after_tx = int(info_after_tx[2])
                finalized_after_tx = bool(ann_after_tx[4])
                stage_elapsed = time.perf_counter() - confirmation_stage_t0

                append_performance_row(
                    stage="confirmation",
                    transaction_type="Member confirmation",
                    community_id=int(community_id),
                    community_size=int(community_size),
                    required_confirmations=int(required_count),
                    confirmation_count_before=int(confirm_count_before_tx),
                    confirmation_count_after=int(confirm_count_after_tx),
                    is_finalizing=bool(finalized_after_tx),
                    stage_elapsed_sec=float(stage_elapsed),
                    sender_address=addr,
                    contract_address=contract_address,
                    note=(
                        "Finalization-triggering confirmation"
                        if finalized_after_tx
                        else "Regular member confirmation"
                    ),
                    **perf,
                )

                print(
                    f"[OK] confirmation | user_id={user_id} | "
                    f"count={confirm_count_after_tx}/{required_count} | "
                    f"tx={last_tx_hash} | gasUsed={perf['gas_used']} | "
                    f"latency={perf['latency_sec']:.6f}s | "
                    f"finalizing={finalized_after_tx}"
                )

            except Exception as e:
                print(f"[WARNING] confirm failed | user_id={user_id} | error={str(e)}")

        # قراءة الحالة النهائية بعد المحاولة
        ann_final = contract.functions.getAnnouncement(int(community_id)).call()
        info_final = contract.functions.getCommunityConfirmationInfo(int(community_id)).call()

        finalized_final = bool(ann_final[4])
        current_confirm_count_final = int(info_final[2])

        if finalized_final:
            status = "finalized"
        elif current_confirm_count_final >= required_count:
            status = "threshold_reached_but_check_needed"
        else:
            status = "not_enough_confirmations"

    # Final read
    ann_after = contract.functions.getAnnouncement(int(community_id)).call()
    timing_after = contract.functions.getAnnouncementTiming(int(community_id)).call()
    info_after = contract.functions.getCommunityConfirmationInfo(int(community_id)).call()

    final_Ak = int(ann_after[0])
    final_confirm_count = int(ann_after[2])
    final_exists = bool(ann_after[3])
    final_finalized = bool(ann_after[4])
    final_expired = bool(timing_after[2])
    final_required = int(info_after[1])
    final_size = int(info_after[0])

    finalizedAk_value = contract.functions.finalizedAk(int(community_id)).call()

    print("[STATE AFTER]")
    print("exists       :", final_exists)
    print("finalized    :", final_finalized)
    print("confirmCount :", final_confirm_count)
    print("required     :", final_required)
    print("finalizedAk  :", finalizedAk_value)
    print("expired      :", final_expired)

    report_rows.append({
        "community_id": int(community_id),
        "A_k": int(final_Ak),
        "status": status,
        "tx_sent": int(tx_sent),
        "last_tx_hash": last_tx_hash,
        "exists": final_exists,
        "finalized": final_finalized,
        "confirm_count": final_confirm_count,
        "required_confirmations": final_required,
        "community_size": final_size,
        "finalizedAk": int(finalizedAk_value),
        "expired": final_expired,
        "deadline": int(timing_after[1]),
    })


# =====================================================================================
# Save report
# =====================================================================================

df_report = pd.DataFrame(report_rows)
df_report.to_csv(CONFIRM_REPORT_OUT, index=False, encoding="utf-8-sig")

print("\n" + "=" * 90)
print("[DONE] Confirmation process completed.")
print("[PATH] Confirm report:", CONFIRM_REPORT_OUT)

print("\n[SUMMARY]")
print(
    df_report[
        [
            "community_id",
            "A_k",
            "status",
            "finalized",
            "confirm_count",
            "required_confirmations",
            "finalizedAk",
            "expired",
            "tx_sent",
        ]
    ].to_string(index=False)
)

[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 750
[INFO] Contract address: 0x27fFA41Dd35f806E2e8b320c1f074B7E33d8B902
[INFO] confirmationWindowSeconds: 3600

[START] Confirming A_k announcements...

[COMMUNITY] 8
[STATE BEFORE]
exists       : True
finalized    : False
A_k          : 378
confirmCount : 1
required     : 53
expired      : False
deadline     : 1788215600
[OK] confirmation | user_id=22 | count=2/53 | tx=0xea9ffdad81bce4c07f8400fdd8cf4001ba526b89135d0ca0e2d51faca979322f | gasUsed=90844 | latency=0.033889s | finalizing=False
[OK] confirmation | user_id=53 | count=3/53 | tx=0x8379333b47de527a0700d54d0f881b92087208c785e03a4a919c28273dc8496a | gasUsed=90844 | latency=0.035518s | finalizing=False
[OK] confirmation | user_id=105 | count=4/53 | tx=0xb39df4e00e3c4c0d004a66725fe8636fed14d6952cfa7529ea0ed9d8606198f8 | gasUsed=90844 | latency=0.032796s | finalizing=False
[OK] confirmation | user_id=124 | count=5/53 | tx=0x95d80572189fb6627ca19375a810c703792a47e238d9e

### الخطوة الثامنة
نعم، الآن نعمل خطوة قراءة النتائج النهائية من العقد وتصدير تقرير نهائي.

هذه الخطوة لا تغيّر شيئًا داخل العقد، فقط تقرأ الحالة النهائية وتنتج ملفًا مرتبًا يثبت أن كل 
A_k
 تم تثبيته. نتائج التأكيد السابقة أظهرت أن كل المجتمعات وصلت إلى 
finalized = True، وكل finalizedAk 
يطابق A_k المطلوب.

In [10]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Read Finalized A_k Results from Smart Contract
# =====================================================================================
#
# الهدف:
# قراءة النتائج النهائية من العقد الذكي بعد مرحلة الإعلان والتأكيد.
#
# يقرأ:
# 1) finalizedAk(communityId)
# 2) getAnnouncement(communityId)
# 3) getAnnouncementTiming(communityId)
# 4) getCommunityConfirmationInfo(communityId)
#
# وينتج:
# finalized_ak_report.csv
#
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

AK_ANNOUNCEMENTS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "chain_announcements"
    / "ak_chain_announcements.csv"
)

ABI_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.abi.json"
)

CONTRACT_ADDRESS_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.address.txt"
)

FINALIZED_REPORT_OUT = (
    ANNOUNCEMENT_ROOT
    / "outputs"
    / "logs"
    / "finalized_ak_report.csv"
)

FINALIZED_REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# Checks
# =====================================================================================

required_files = [
    AK_ANNOUNCEMENTS_CSV,
    ABI_PATH,
    CONTRACT_ADDRESS_PATH,
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")


# =====================================================================================
# Connect to Ganache and Contract
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))

abi = json.loads(ABI_PATH.read_text(encoding="utf-8"))
contract_address = CONTRACT_ADDRESS_PATH.read_text(encoding="utf-8").strip()
contract_address = Web3.to_checksum_address(contract_address)

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

print("[INFO] Contract address:", contract_address)
print("[INFO] confirmationWindowSeconds:", contract.functions.confirmationWindowSeconds().call())


# =====================================================================================
# Load expected A_k values
# =====================================================================================

df_expected = pd.read_csv(AK_ANNOUNCEMENTS_CSV)

df_expected["community_id"] = df_expected["community_id"].astype(int)
df_expected["A_k"] = df_expected["A_k"].astype(int)

community_ids = sorted(df_expected["community_id"].unique().tolist())


# =====================================================================================
# Read final state from contract
# =====================================================================================

rows = []

print("\n[START] Reading finalized A_k values from contract...")

for community_id in community_ids:
    expected_row = df_expected[df_expected["community_id"] == community_id].iloc[0]

    expected_Ak = int(expected_row["A_k"])
    expected_required = int(expected_row["required_confirmations"])
    expected_hash = str(expected_row["result_hash"]).strip()

    # getAnnouncement returns:
    # Ak, resultHash, confirmCount, exists, finalized, announcer
    ann = contract.functions.getAnnouncement(int(community_id)).call()

    # getAnnouncementTiming returns:
    # announcedAt, deadline, expired
    timing = contract.functions.getAnnouncementTiming(int(community_id)).call()

    # getCommunityConfirmationInfo returns:
    # size, requiredCount, currentConfirmCount
    info = contract.functions.getCommunityConfirmationInfo(int(community_id)).call()

    finalized_Ak = int(contract.functions.finalizedAk(int(community_id)).call())

    onchain_Ak = int(ann[0])
    onchain_result_hash = ann[1].hex()
    confirm_count = int(ann[2])
    exists = bool(ann[3])
    finalized = bool(ann[4])
    announcer = ann[5]

    announced_at = int(timing[0])
    deadline = int(timing[1])
    expired = bool(timing[2])

    community_size = int(info[0])
    required_count = int(info[1])
    current_confirm_count = int(info[2])

    Ak_matches = (expected_Ak == finalized_Ak == onchain_Ak)
    required_matches = (expected_required == required_count)
    finalized_ok = finalized and (confirm_count >= required_count)
    hash_matches = (
        expected_hash.lower().replace("0x", "") ==
        onchain_result_hash.lower().replace("0x", "")
    )

    if finalized_ok and Ak_matches and required_matches and hash_matches:
        validation_status = "valid_finalized"
    else:
        validation_status = "check_needed"

    print("\n" + "=" * 90)
    print(f"[COMMUNITY] {community_id}")
    print("expected_Ak     :", expected_Ak)
    print("onchain_Ak      :", onchain_Ak)
    print("finalized_Ak    :", finalized_Ak)
    print("finalized       :", finalized)
    print("confirm_count   :", confirm_count)
    print("required_count  :", required_count)
    print("expired         :", expired)
    print("hash_matches    :", hash_matches)
    print("status          :", validation_status)

    rows.append({
        "community_id": int(community_id),

        "expected_Ak": expected_Ak,
        "onchain_Ak": onchain_Ak,
        "finalized_Ak": finalized_Ak,

        "exists": exists,
        "finalized": finalized,
        "expired": expired,

        "community_size": community_size,
        "confirm_count": confirm_count,
        "required_confirmations": required_count,
        "expected_required_confirmations": expected_required,

        "announcer_address": announcer,

        "expected_result_hash": expected_hash,
        "onchain_result_hash": onchain_result_hash,

        "Ak_matches": Ak_matches,
        "required_matches": required_matches,
        "hash_matches": hash_matches,
        "finalized_ok": finalized_ok,

        "announced_at": announced_at,
        "deadline": deadline,

        "validation_status": validation_status,
    })


# =====================================================================================
# Save report
# =====================================================================================

df_report = pd.DataFrame(rows)
df_report.to_csv(FINALIZED_REPORT_OUT, index=False, encoding="utf-8-sig")

print("\n" + "=" * 90)
print("[DONE] Finalized A_k report generated.")
print("[PATH] Finalized report:", FINALIZED_REPORT_OUT)

print("\n[SUMMARY]")
print(
    df_report[
        [
            "community_id",
            "expected_Ak",
            "finalized_Ak",
            "finalized",
            "confirm_count",
            "required_confirmations",
            "expired",
            "Ak_matches",
            "hash_matches",
            "validation_status",
        ]
    ].to_string(index=False)
)

[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 750
[INFO] Contract address: 0x27fFA41Dd35f806E2e8b320c1f074B7E33d8B902
[INFO] confirmationWindowSeconds: 3600

[START] Reading finalized A_k values from contract...

[COMMUNITY] 8
expected_Ak     : 378
onchain_Ak      : 378
finalized_Ak    : 378
finalized       : True
confirm_count   : 53
required_count  : 53
expired         : False
hash_matches    : True
status          : valid_finalized

[COMMUNITY] 14
expected_Ak     : 289
onchain_Ak      : 289
finalized_Ak    : 289
finalized       : True
confirm_count   : 69
required_count  : 69
expired         : False
hash_matches    : True
status          : valid_finalized

[COMMUNITY] 84
expected_Ak     : 28
onchain_Ak      : 28
finalized_Ak    : 28
finalized       : True
confirm_count   : 59
required_count  : 59
expired         : False
hash_matches    : True
status          : valid_finalized

[COMMUNITY] 96
expected_Ak     : 30
onchain_Ak      : 30
finalized_Ak    : 30
finalized 

### ملخص قياسات الأداء — يولد تلقائياً بعد اكتمال التثبيت

هذه الخلية تجمع السجل الخام للمعاملات وتنتج ملخصاً حسب المرحلة وملخصاً حسب المجتمع. تفصل كلفة النشر والتحميل عن كلفة التخويل والتثبيت.


In [11]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Blockchain Performance Summary — Douban Movie
# =====================================================================================

import numpy as np
import pandas as pd

if not PERF_TX_CSV.exists():
    raise FileNotFoundError(
        f"Performance transaction log not found: {PERF_TX_CSV}. "
        "Run the measured blockchain stages first."
    )

df_perf = pd.read_csv(PERF_TX_CSV)
for col in ["gas_used", "latency_sec", "cost_eth", "receipt_status", "community_size", "required_confirmations", "stage_elapsed_sec"]:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors="coerce")

stage_rows = []
for (stage, tx_type), g in df_perf.groupby(["stage", "transaction_type"], dropna=False):
    total_latency = float(g["latency_sec"].sum())
    successful = int((g["receipt_status"] == 1).sum())
    stage_rows.append({
        "stage": stage,
        "transaction_type": tx_type,
        "transactions": int(len(g)),
        "successful_transactions": successful,
        "total_gas_used": int(g["gas_used"].sum()),
        "mean_gas_used": float(g["gas_used"].mean()),
        "median_gas_used": float(g["gas_used"].median()),
        "min_gas_used": int(g["gas_used"].min()),
        "max_gas_used": int(g["gas_used"].max()),
        "total_latency_sec": total_latency,
        "mean_latency_sec": float(g["latency_sec"].mean()),
        "throughput_tx_per_sec_serial": (successful / total_latency) if total_latency > 0 else np.nan,
        "total_cost_eth_ganache": float(g["cost_eth"].sum()),
    })

df_stage_summary = pd.DataFrame(stage_rows)
df_stage_summary.to_csv(PERF_STAGE_SUMMARY_CSV, index=False, encoding="utf-8-sig")

auth = df_perf[df_perf["stage"].isin(["announcement", "confirmation"])].copy()
community_rows = []

for community_id, g in auth.groupby("community_id"):
    g_ann = g[g["stage"] == "announcement"]
    g_conf = g[g["stage"] == "confirmation"]
    final_mask = g_conf["is_finalizing"].astype(str).str.lower().isin(["true", "1"])
    g_final = g_conf[final_mask]

    community_size = int(pd.to_numeric(g["community_size"], errors="coerce").dropna().max())
    required = int(pd.to_numeric(g["required_confirmations"], errors="coerce").dropna().max())
    announcement_latency = float(g_ann["latency_sec"].sum())
    confirmation_stage_elapsed = (
        float(pd.to_numeric(g_conf["stage_elapsed_sec"], errors="coerce").max())
        if len(g_conf) else 0.0
    )
    controlled_processing_time = announcement_latency + confirmation_stage_elapsed
    successful = int((g["receipt_status"] == 1).sum())

    community_rows.append({
        "community_id": int(community_id),
        "community_size": community_size,
        "required_confirmations": required,
        "announcement_transactions": int(len(g_ann)),
        "confirmation_transactions": int(len(g_conf)),
        "authorization_transactions_total": int(len(g)),
        "announcement_gas_used": int(g_ann["gas_used"].sum()),
        "confirmation_gas_used": int(g_conf["gas_used"].sum()),
        "authorization_gas_used_total": int(g["gas_used"].sum()),
        "mean_confirmation_gas_used": float(g_conf["gas_used"].mean()) if len(g_conf) else np.nan,
        "finalizing_confirmation_gas_used": int(g_final["gas_used"].iloc[-1]) if len(g_final) else np.nan,
        "announcement_latency_sec": announcement_latency,
        "mean_confirmation_latency_sec": float(g_conf["latency_sec"].mean()) if len(g_conf) else np.nan,
        "confirmation_stage_elapsed_sec": confirmation_stage_elapsed,
        "controlled_authorization_processing_time_sec": controlled_processing_time,
        "authorization_throughput_tx_per_sec": (successful / controlled_processing_time) if controlled_processing_time > 0 else np.nan,
        "authorization_cost_eth_ganache": float(g["cost_eth"].sum()),
        "finalization_reached": bool(len(g_final) > 0),
    })

df_community_summary = pd.DataFrame(community_rows).sort_values("community_id")
df_community_summary.to_csv(PERF_COMMUNITY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

update_run_metadata(
    completed_at_utc=utc_now_iso(),
    ending_block=int(w3.eth.block_number) if w3.is_connected() else None,
    transaction_log=str(PERF_TX_CSV),
    stage_summary=str(PERF_STAGE_SUMMARY_CSV),
    community_summary=str(PERF_COMMUNITY_SUMMARY_CSV),
    measured_transactions=int(len(df_perf)),
)

print("=" * 100)
print("[PERFORMANCE SUMMARY — BY STAGE]")
print("=" * 100)
print(df_stage_summary.to_string(index=False))
print("\n" + "=" * 100)
print("[PERFORMANCE SUMMARY — BY COMMUNITY]")
print("=" * 100)
print(df_community_summary.to_string(index=False))
print("\n[FILES]")
print("Raw transactions :", PERF_TX_CSV)
print("By stage         :", PERF_STAGE_SUMMARY_CSV)
print("By community     :", PERF_COMMUNITY_SUMMARY_CSV)
print("Run metadata     :", PERF_RUN_META_JSON)


[PERFORMANCE SUMMARY — BY STAGE]
         stage             transaction_type  transactions  successful_transactions  total_gas_used  mean_gas_used  median_gas_used  min_gas_used  max_gas_used  total_latency_sec  mean_latency_sec  throughput_tx_per_sec_serial  total_cost_eth_ganache
  announcement Authorized-node announcement             6                        6         1405962   2.343270e+05         234325.0        234319        234343           0.236502          0.039417                     25.369796                0.000111
  confirmation          Member confirmation           345                      345        31519878   9.136197e+04          90844.0         90832        120853          11.483582          0.033286                     30.042892                0.000035
    deployment          Contract deployment             1                        1         1271695   1.271695e+06        1271695.0       1271695       1271695           0.073753          0.073753                     1

# هذا لم نقم بأضافته لانه فيه تغيير الهاش الخاص بالاعلان يفتح اسئلة في غنى عنها
لا يعتمد
الخلية اللي بعدها نعتمدها

### تم. أضفت Cell 14 — اختبار تلاعب اختياري إلى نسخة جديدة من ملف الإعلان/التثبيت.

الاختبار يعمل بنفس فكرة ملف إيصال التوصيات، لكن على مرحلة إعلان وتثبيت A_k:

يقرأ سجل إعلان واحد من ملفات ak_announcement_comm_*.json.
يحسب result_hash الأصلي من جديد.
يقارنه مع result_hash المثبّت على العقد.
يغيّر قيمة A_k داخل الذاكرة فقط، بدون تعديل أي ملف.
يعيد حساب الهاش بعد التلاعب.
يتأكد أن الهاش الجديد لا يطابق الهاش الموجود على العقد.
يحاكي محاولة إعادة الإعلان باستخدام .call() فقط، بدون transact، حتى لا يغيّر حالة البلوكشين.

النتيجة المتوقعة تكون:

PASS إذا كان الأصل مطابقًا، والتلاعب مرفوضًا بالمقارنة مع الهاش المثبّت على العقد.

In [12]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 14 — Optional Tampering Test for A_k Announcement Integrity
# =====================================================================================
#
# الهدف:
# اختبار ماذا يحدث إذا قام شخص بتغيير نتيجة الإعلان خارج السلسلة off-chain.
#
# هذا الاختبار:
# 1) لا يغيّر ملفات JSON أو CSV.
# 2) لا يرسل transact ولا يغيّر حالة العقد.
# 3) يغيّر نسخة من payload داخل الذاكرة فقط.
# 4) يعيد حساب result_hash بعد التلاعب.
# 5) يقارن الهاش الجديد مع الهاش المثبّت على العقد.
# 6) يحاكي محاولة إعادة الإعلان باستخدام call فقط، وليس transact.
#
# Expected behavior:
# - original hash يجب أن يطابق الهاش المثبّت على العقد.
# - tampered hash يجب ألا يطابق الهاش المثبّت على العقد.
# - simulated re-announcement يجب أن يُرفض إذا كان الإعلان موجودًا أو مثبتًا.
# =====================================================================================

from pathlib import Path
import json
import hashlib
import copy

import pandas as pd
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

AK_ANNOUNCEMENTS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "chain_announcements"
    / "ak_chain_announcements.csv"
)

CHAIN_ANNOUNCEMENTS_DIR = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "chain_announcements"
)

COMMUNITY_MEMBERS_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "communities"
    / "community_members.csv"
)

ADDRESS_TO_USERID_CSV = (
    ANNOUNCEMENT_ROOT
    / "data"
    / "blockchain"
    / "address_to_userid.csv"
)

ABI_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.abi.json"
)

CONTRACT_ADDRESS_PATH = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "build"
    / "AuthorizedNodeElection.address.txt"
)


# =====================================================================================
# Helper functions
# =====================================================================================

def normalize_hash_hex(value: str) -> str:
    """Normalize a hex hash by removing 0x and converting to lowercase."""
    h = str(value).strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return h


def stable_json_hash(payload: dict) -> str:
    """
    Recompute the same SHA-256 hash style used when the announcement JSON was created.
    """
    normalized = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def remove_non_hash_fields(payload: dict) -> dict:
    """
    The original result_hash was calculated before adding result_hash and created_at.
    Therefore, these fields must be removed before recomputing the hash.
    """
    cleaned = copy.deepcopy(payload)
    cleaned.pop("result_hash", None)
    cleaned.pop("created_at", None)
    return cleaned


def result_hash_to_bytes32(result_hash_hex: str) -> bytes:
    """Convert a 64-character hex hash into bytes32 for the contract call simulation."""
    h = normalize_hash_hex(result_hash_hex)
    if len(h) != 64:
        raise ValueError(f"Invalid result_hash length. Expected 64 hex chars, got {len(h)}")
    return bytes.fromhex(h)


def find_announcement_json(row: pd.Series) -> Path:
    """Get announcement JSON path from csv if available; otherwise build it from community_id."""
    community_id = int(row["community_id"])

    if "json_file" in row.index and pd.notna(row["json_file"]):
        json_path = Path(str(row["json_file"]))
        if json_path.exists():
            return json_path

    json_path = CHAIN_ANNOUNCEMENTS_DIR / f"ak_announcement_comm_{community_id}.json"
    if json_path.exists():
        return json_path

    raise FileNotFoundError(
        f"Announcement JSON file not found for community={community_id}. "
        f"Checked csv json_file and: {json_path}"
    )


def select_announcer_address(community_id: int) -> tuple[int, str]:
    """
    Select the first community member address, same idea used in the announcement cell.
    This is only for .call simulation; no transaction is sent.
    """
    df_members = pd.read_csv(COMMUNITY_MEMBERS_CSV)
    df_addr = pd.read_csv(ADDRESS_TO_USERID_CSV)

    df_members["community_id"] = df_members["community_id"].astype(int)
    df_members["user_id"] = df_members["user_id"].astype(int)

    df_addr["user_id"] = df_addr["user_id"].astype(int)
    df_addr["address"] = df_addr["address"].astype(str)

    user_to_address = dict(
        zip(
            df_addr["user_id"].astype(int),
            df_addr["address"].astype(str)
        )
    )

    users = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    if not users:
        raise RuntimeError(f"No members found for community={community_id}")

    announcer_user_id = int(users[0])

    if announcer_user_id not in user_to_address:
        raise RuntimeError(f"Missing address for user_id={announcer_user_id}")

    return announcer_user_id, Web3.to_checksum_address(user_to_address[announcer_user_id])


# =====================================================================================
# Checks
# =====================================================================================

required_files = [
    AK_ANNOUNCEMENTS_CSV,
    COMMUNITY_MEMBERS_CSV,
    ADDRESS_TO_USERID_CSV,
    ABI_PATH,
    CONTRACT_ADDRESS_PATH,
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")


# =====================================================================================
# Connect to Ganache and contract
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

abi = json.loads(ABI_PATH.read_text(encoding="utf-8"))
contract_address = Web3.to_checksum_address(
    CONTRACT_ADDRESS_PATH.read_text(encoding="utf-8").strip()
)

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

print("[OK] Connected to contract:", contract_address)


# =====================================================================================
# Select one announcement record for tampering test
# =====================================================================================

df_ak = pd.read_csv(AK_ANNOUNCEMENTS_CSV)
df_ak["community_id"] = df_ak["community_id"].astype(int)
df_ak["A_k"] = df_ak["A_k"].astype(int)

selected_row = None
selected_json_path = None

for _, row in df_ak.sort_values("community_id").iterrows():
    community_id = int(row["community_id"])
    ann = contract.functions.getAnnouncement(community_id).call()
    exists = bool(ann[3])

    if exists:
        selected_row = row
        selected_json_path = find_announcement_json(row)
        break

if selected_row is None:
    raise RuntimeError(
        "No existing on-chain announcement was found. "
        "Run the announcement cell first, then run this tampering test."
    )

community_id = int(selected_row["community_id"])

with selected_json_path.open("r", encoding="utf-8") as f:
    original_file_payload = json.load(f)

original_hash_payload = remove_non_hash_fields(original_file_payload)
recomputed_original_hash = stable_json_hash(original_hash_payload)

ann = contract.functions.getAnnouncement(community_id).call()
onchain_Ak = int(ann[0])
onchain_result_hash = normalize_hash_hex(ann[1].hex())
confirm_count = int(ann[2])
exists = bool(ann[3])
finalized = bool(ann[4])
announcer_onchain = ann[5]

expected_hash_from_file = normalize_hash_hex(
    original_file_payload.get("result_hash", selected_row.get("result_hash", ""))
)

print("\n" + "=" * 90)
print("[ORIGINAL RECORD]")
print("community_id          :", community_id)
print("json_path             :", selected_json_path)
print("A_k in file           :", int(original_hash_payload["A_k"]))
print("A_k on-chain          :", onchain_Ak)
print("exists on-chain       :", exists)
print("finalized on-chain    :", finalized)
print("confirm_count         :", confirm_count)
print("announcer on-chain    :", announcer_onchain)
print("file result_hash      :", expected_hash_from_file)
print("recomputed hash       :", recomputed_original_hash)
print("on-chain result_hash  :", onchain_result_hash)

original_matches_file = (
    normalize_hash_hex(recomputed_original_hash) == expected_hash_from_file
)

original_matches_onchain = (
    normalize_hash_hex(recomputed_original_hash) == onchain_result_hash
)

print("\n[ORIGINAL HASH CHECK]")
print("recomputed hash matches file    :", original_matches_file)
print("recomputed hash matches on-chain:", original_matches_onchain)


# =====================================================================================
# Tamper in memory only
# =====================================================================================

# Deep copy: no file is modified.
tampered_payload = copy.deepcopy(original_hash_payload)

# تغيير مقصود داخل الذاكرة فقط.
# يمكنك تغيير حقل آخر إذا أردت، لكن تغيير A_k يوضح محاولة تبديل القائد المختار.
original_Ak = int(tampered_payload["A_k"])
tampered_payload["A_k"] = original_Ak + 999999

tampered_hash = stable_json_hash(tampered_payload)

tampered_hash_matches_onchain = (
    normalize_hash_hex(tampered_hash) == onchain_result_hash
)

tampered_Ak_matches_onchain = (
    int(tampered_payload["A_k"]) == onchain_Ak
)

print("\n" + "=" * 90)
print("[TAMPERED IN-MEMORY RECORD]")
print("original A_k          :", original_Ak)
print("tampered A_k          :", int(tampered_payload["A_k"]))
print("tampered hash         :", tampered_hash)
print("tampered A_k matches on-chain  :", tampered_Ak_matches_onchain)
print("tampered hash matches on-chain :", tampered_hash_matches_onchain)

if not tampered_hash_matches_onchain:
    print("[EXPECTED] REJECTED by hash comparison: tampered content does not match the on-chain hash reference.")
else:
    print("[UNEXPECTED] The tampered hash matched the on-chain hash. Check hashing logic.")


# =====================================================================================
# Simulate re-announcement with tampered values using call only
# =====================================================================================

announcer_user_id, announcer_address = select_announcer_address(community_id)

print("\n" + "=" * 90)
print("[SIMULATED RE-ANNOUNCEMENT — call only, no transaction]")
print("simulated sender user_id :", announcer_user_id)
print("simulated sender address :", announcer_address)

simulated_reannouncement_rejected = False
simulation_error = None

try:
    # Important: .call() simulates execution and does not modify blockchain state.
    contract.functions.announceAk(
        int(community_id),
        int(tampered_payload["A_k"]),
        result_hash_to_bytes32(tampered_hash)
    ).call({"from": announcer_address})

    simulated_reannouncement_rejected = False
    print("[UNEXPECTED] Simulated tampered re-announcement did not revert.")
    print("This may happen only if no announcement exists; check the previous cells and contract state.")

except Exception as e:
    simulated_reannouncement_rejected = True
    simulation_error = str(e)
    print("[EXPECTED] Simulated tampered re-announcement was rejected by the contract.")
    print("Reason preview:", simulation_error[:300])


# =====================================================================================
# Final compact result
# =====================================================================================

summary = pd.DataFrame([
    {
        "community_id": community_id,
        "original_Ak": original_Ak,
        "onchain_Ak": onchain_Ak,
        "original_hash_matches_file": original_matches_file,
        "original_hash_matches_onchain": original_matches_onchain,
        "tampered_Ak": int(tampered_payload["A_k"]),
        "tampered_Ak_matches_onchain": tampered_Ak_matches_onchain,
        "tampered_hash_matches_onchain": tampered_hash_matches_onchain,
        "simulated_reannouncement_rejected": simulated_reannouncement_rejected,
        "expected_result": "PASS" if (
            original_matches_file
            and original_matches_onchain
            and not tampered_Ak_matches_onchain
            and not tampered_hash_matches_onchain
            and simulated_reannouncement_rejected
        ) else "CHECK_NEEDED"
    }
])

print("\n" + "=" * 90)
print("[TAMPERING TEST SUMMARY]")
print(summary.to_string(index=False))

summary

An RPC error was returned by the node. Check the message provided in the error and any available logs for more information.


[OK] Connected to contract: 0x27fFA41Dd35f806E2e8b320c1f074B7E33d8B902

[ORIGINAL RECORD]
community_id          : 8
json_path             : D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان\data\chain_announcements\ak_announcement_comm_8.json
A_k in file           : 378
A_k on-chain          : 378
exists on-chain       : True
finalized on-chain    : True
confirm_count         : 53
announcer on-chain    : 0x7F6740D24949E7E77EaB673Cf4A651F09bc7BfdD
file result_hash      : 5b391579cb21cafda4b1d09bfdbb2853c05ec738c9be421d1294eadfed9f5d6a
recomputed hash       : 5b391579cb21cafda4b1d09bfdbb2853c05ec738c9be421d1294eadfed9f5d6a
on-chain result_hash  : 5b391579cb21cafda4b1d09bfdbb2853c05ec738c9be421d1294eadfed9f5d6a

[ORIGINAL HASH CHECK]
recomputed hash matches file    : True
recomputed hash matches on-chain: True

[TAMPERED IN-MEMORY RECORD]
original A_k          : 378
tampered A_k          : 1000377
tampered hash         : da795a073781f6a024f2e1ad861b748878ff8d8ad8721b5e3e38

,community_id,original_Ak,onchain_Ak,original_hash_matches_file,original_hash_matches_onchain,tampered_Ak,tampered_Ak_matches_onchain,tampered_hash_matches_onchain,simulated_reannouncement_rejected,expected_result
0,8,378,378,True,True,1000377,False,False,True,PASS


# معتمد و لايحتاج الى تنفيذ الخطوات السابقة# 
### اختبار التوقيت عندما يصبح اكسباير
الفكرة العملية:

ننشئ عقد مؤقت للاختبار فقط بوقت قصير مثل 10 ثواني، نعلن A_k لمجتمع تجريبي صغير، ننتظر 12 ثانية، ثم نحاول التأكيد. إذا رفض العقد التأكيد بعد انتهاء الوقت، فالاختبار ناجح.

ضع هذه الخلية بعد خلايا النشر أو في نهاية الـNotebook كاختبار مستقل.

نعم، كلامك صحيح. عدم وجود تصويت/تأكيد قد يعني أحد سببين:

الحالة	المعنى
Delay / no response	النودات لم ترسل التأكيد خلال المهلة، بسبب تأخير أو عدم مشاركة
### Invalid announced Aₖ	النودات حسبت النتيجة ووجدت أن
Aₖ
المعلنة ليست هي الـ
trusted node
الصحيحة، لذلك امتنعت عن التأكيد

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 15 — Invalid Announcement and Confirmation Deadline Test
# =====================================================================================
#
# الهدف:
# اختبار حالة إعلان A_k غير صحيحة ثم عدم تأكيدها من قبل النودات الصادقة
# إلى أن تنتهي نافذة التأكيد.
#
# الفكرة:
# 1) ننشر عقدًا مؤقتًا بمهلة قصيرة، مثل 10 ثوانٍ.
# 2) نحمّل مجتمعًا تجريبيًا صغيرًا.
# 3) نحدد A_k الصحيحة المفترضة حسب الحساب خارج السلسلة: EXPECTED_AK.
# 4) نعلن A_k خاطئة: ANNOUNCED_AK.
# 5) النودات الصادقة تقارن ANNOUNCED_AK مع EXPECTED_AK.
# 6) إذا لم تتطابق، النودات لا ترسل confirmAk.
# 7) نحرّك وقت Ganache ونعدين بلوك جديد حتى تتحدّث block.timestamp.
# 8) بعد انتهاء المهلة يصبح الإعلان expired وغير finalized.
# 9) محاولة التأكيد بعد انتهاء المهلة يجب أن تُرفض.
#
# هذا الاختبار:
# - لا يغيّر العقد الأصلي المستخدم في النتائج.
# - لا يغيّر ملفات النتائج الأصلية.
# - لا يستبدل عنوان العقد الأصلي المحفوظ.
# - يستخدم عقدًا مؤقتًا فقط لغرض الاختبار.
#
# Expected behavior:
# - ANNOUNCED_AK != EXPECTED_AK
# - honest_nodes_decision = SKIP_CONFIRMATION
# - confirm_count يبقى أقل من required_confirmations
# - expired_after_wait = True
# - finalized_after_wait = False
# - late confirm after deadline = Rejected
# - expected_result = PASS
# =====================================================================================

from pathlib import Path
import json
import time
import hashlib
import pandas as pd

from solcx import compile_source, install_solc
from web3 import Web3


# =====================================================================================
# Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\3 الاعلان"
)

RPC_URL = "http://127.0.0.1:7545"

CONTRACT_FILE = (
    ANNOUNCEMENT_ROOT
    / "contracts"
    / "AuthorizedNodeElection.sol"
)

INVALID_TIMING_TEST_REPORT_OUT = (
    ANNOUNCEMENT_ROOT
    / "outputs"
    / "logs"
    / "invalid_announcement_timing_test_report.csv"
)

INVALID_TIMING_TEST_REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# Test settings
# =====================================================================================

# مهلة قصيرة حتى نحصل على النتيجة بسرعة
TEST_CONFIRMATION_WINDOW_SECONDS = 10

# نزيد وقت البلوكشين بأكثر من نافذة التأكيد حتى نضمن انتهاء المهلة
WAIT_SECONDS = TEST_CONFIRMATION_WINDOW_SECONDS + 2

# مجتمع تجريبي وهمي لا علاقة له بنتائج Douban الأصلية
TEST_COMMUNITY_ID = 999001

# A_k الصحيحة المفترضة حسب إعادة الحساب خارج السلسلة
EXPECTED_AK = 123456

# A_k الخاطئة التي سيتم إعلانها لاختبار رفض التأكيد من النودات الصادقة
ANNOUNCED_AK = 999999

TX_GAS_DEPLOY = 7_000_000
TX_GAS_LOAD = 1_000_000
TX_GAS_ANNOUNCE = 700_000
TX_GAS_CONFIRM = 300_000


# =====================================================================================
# Helper functions
# =====================================================================================

def ganache_rpc(provider, method, params=None):
    """
    Execute a Ganache JSON-RPC method and raise an error if Ganache returns an error.
    """
    if params is None:
        params = []

    response = provider.make_request(method, params)

    if isinstance(response, dict) and response.get("error"):
        raise RuntimeError(f"{method} failed: {response['error']}")

    return response


def advance_ganache_time(w3, seconds, owner_address):
    """
    Advance Ganache blockchain time and mine a new block.
    
    Why?
    time.sleep() waits in Python, but block.timestamp in Ganache changes only when
    a new block is mined. Therefore, we use evm_increaseTime + evm_mine.
    If these methods fail, we fall back to real sleep + dummy transaction.
    """
    try:
        ganache_rpc(w3.provider, "evm_increaseTime", [int(seconds)])
        ganache_rpc(w3.provider, "evm_mine", [])
        print("[OK] Ganache time advanced and one block mined.")

    except Exception as e:
        print("[WARN] evm_increaseTime/evm_mine failed.")
        print("[WARN] Falling back to real sleep + dummy transaction.")
        print("Reason preview:", str(e)[:300])

        time.sleep(int(seconds))

        dummy_tx = w3.eth.send_transaction({
            "from": owner_address,
            "to": owner_address,
            "value": 0,
            "gas": 21_000,
        })

        w3.eth.wait_for_transaction_receipt(dummy_tx)
        print("[OK] Dummy transaction mined to refresh block.timestamp.")


# =====================================================================================
# Checks
# =====================================================================================

if not CONTRACT_FILE.exists():
    raise FileNotFoundError(f"Contract file not found: {CONTRACT_FILE}")

if int(ANNOUNCED_AK) == int(EXPECTED_AK):
    raise ValueError("ANNOUNCED_AK must be different from EXPECTED_AK for this test.")


# =====================================================================================
# Connect to Ganache
# =====================================================================================

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise ConnectionError(f"Could not connect to Ganache at {RPC_URL}")

accounts = w3.eth.accounts

if len(accounts) < 4:
    raise RuntimeError("Ganache must provide at least 4 accounts for this test.")

owner_address = Web3.to_checksum_address(accounts[0])

# ثلاثة أعضاء تجريبيين فقط
test_members = [
    Web3.to_checksum_address(accounts[1]),
    Web3.to_checksum_address(accounts[2]),
    Web3.to_checksum_address(accounts[3]),
]

announcer_address = test_members[0]
honest_validator_address = test_members[1]

print("[OK] Connected to Ganache")
print("[INFO] Owner address       :", owner_address)
print("[INFO] Test members        :", test_members)
print("[INFO] Test community_id   :", TEST_COMMUNITY_ID)
print("[INFO] Test window seconds :", TEST_CONFIRMATION_WINDOW_SECONDS)
print("[INFO] WAIT_SECONDS        :", WAIT_SECONDS)
print("[INFO] EXPECTED_AK         :", EXPECTED_AK)
print("[INFO] ANNOUNCED_AK        :", ANNOUNCED_AK)


# =====================================================================================
# Compile the same AuthorizedNodeElection contract
# =====================================================================================

print("\n[STEP 1] Compile contract...")

install_solc("0.8.20")

source_code = CONTRACT_FILE.read_text(encoding="utf-8")

compiled = compile_source(
    source_code,
    output_values=["abi", "bin"],
    solc_version="0.8.20",
    optimize=True,
)

contract_id, contract_interface = compiled.popitem()

abi = contract_interface["abi"]
bytecode = contract_interface["bin"]

print("[OK] Contract compiled")
print("[INFO] Contract ID:", contract_id)


# =====================================================================================
# Deploy temporary test contract
# =====================================================================================

print("\n[STEP 2] Deploy temporary test contract...")

Contract = w3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

tx_hash = Contract.constructor(TEST_CONFIRMATION_WINDOW_SECONDS).transact({
    "from": owner_address,
    "gas": TX_GAS_DEPLOY,
})

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

if receipt.status != 1:
    raise RuntimeError("Temporary contract deployment failed.")

test_contract_address = Web3.to_checksum_address(receipt.contractAddress)

test_contract = w3.eth.contract(
    address=test_contract_address,
    abi=abi
)

print("[OK] Temporary contract deployed")
print("[INFO] Temporary contract address:", test_contract_address)
print(
    "[INFO] confirmationWindowSeconds:",
    test_contract.functions.confirmationWindowSeconds().call()
)


# =====================================================================================
# Load small test community
# =====================================================================================

print("\n[STEP 3] Load small test community...")

tx_hash = test_contract.functions.loadCommunityMembers(
    int(TEST_COMMUNITY_ID),
    test_members
).transact({
    "from": owner_address,
    "gas": TX_GAS_LOAD,
})

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

if receipt.status != 1:
    raise RuntimeError("Loading test community members failed.")

community_size = int(test_contract.functions.communitySize(TEST_COMMUNITY_ID).call())
required_confirmations = int(
    test_contract.functions.requiredConfirmations(TEST_COMMUNITY_ID).call()
)

print("[OK] Test community loaded")
print("[INFO] community_size         :", community_size)
print("[INFO] required_confirmations :", required_confirmations)


# =====================================================================================
# Announce wrong A_k
# =====================================================================================

print("\n[STEP 4] Announce wrong A_k...")

# هذا payload يمثل ما تم إعلانه، أي يحتوي A_k الخاطئة
announced_payload = {
    "test": "invalid announcement and confirmation deadline test",
    "community_id": int(TEST_COMMUNITY_ID),
    "expected_A_k": int(EXPECTED_AK),
    "announced_A_k": int(ANNOUNCED_AK),
    "window_seconds": int(TEST_CONFIRMATION_WINDOW_SECONDS),
}

announced_hash_hex = hashlib.sha256(
    json.dumps(
        announced_payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    ).encode("utf-8")
).hexdigest()

announced_hash_bytes32 = bytes.fromhex(announced_hash_hex)

tx_hash = test_contract.functions.announceAk(
    int(TEST_COMMUNITY_ID),
    int(ANNOUNCED_AK),
    announced_hash_bytes32
).transact({
    "from": announcer_address,
    "gas": TX_GAS_ANNOUNCE,
})

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

if receipt.status != 1:
    raise RuntimeError("Wrong A_k announcement failed in test contract.")

timing_before_wait = test_contract.functions.getAnnouncementTiming(
    int(TEST_COMMUNITY_ID)
).call()

ann_before_wait = test_contract.functions.getAnnouncement(
    int(TEST_COMMUNITY_ID)
).call()

announced_at_before = int(timing_before_wait[0])
deadline_before = int(timing_before_wait[1])
expired_before_wait = bool(timing_before_wait[2])

onchain_announced_AK = int(ann_before_wait[0])
onchain_result_hash = ann_before_wait[1].hex()
confirm_count_before = int(ann_before_wait[2])
exists_before = bool(ann_before_wait[3])
finalized_before = bool(ann_before_wait[4])
announcer_onchain = ann_before_wait[5]

print("[OK] Wrong A_k announced")
print("[STATE BEFORE VALIDATION]")
print("expected_A_k           :", EXPECTED_AK)
print("announced_A_k          :", ANNOUNCED_AK)
print("on-chain announced_A_k :", onchain_announced_AK)
print("exists                 :", exists_before)
print("finalized              :", finalized_before)
print("confirm_count          :", confirm_count_before)
print("required_confirmations :", required_confirmations)
print("expired                :", expired_before_wait)
print("announced_at           :", announced_at_before)
print("deadline               :", deadline_before)
print("announcer_onchain      :", announcer_onchain)
print("announced_hash         :", announced_hash_hex)
print("onchain_hash           :", onchain_result_hash)


# =====================================================================================
# Honest off-chain validation
# =====================================================================================

print("\n[STEP 5] Honest off-chain validation...")

# النودات الصادقة تعيد حساب نتيجة التصويت خارج السلسلة.
# هنا نمثل النتيجة الصحيحة بالمتغير EXPECTED_AK.
# إذا كانت النتيجة المعلنة لا تطابق النتيجة المتوقعة، لا يتم إرسال confirmAk.

ak_matches_expected = int(ANNOUNCED_AK) == int(EXPECTED_AK)

if ak_matches_expected:
    honest_nodes_decision = "CONFIRM"
    print("[VALIDATION] Announced A_k matches expected A_k.")
    print("[VALIDATION] Honest nodes may submit confirmAk().")
else:
    honest_nodes_decision = "SKIP_CONFIRMATION"
    print("[VALIDATION] Announced A_k does not match expected A_k.")
    print("[VALIDATION] Honest nodes skip confirmAk().")

# في هذا الاختبار نحن نقصد حالة عدم التطابق، لذلك لا نرسل confirmAk قبل انتهاء الوقت.
if honest_nodes_decision == "CONFIRM":
    tx_hash = test_contract.functions.confirmAk(
        int(TEST_COMMUNITY_ID),
        int(ANNOUNCED_AK)
    ).transact({
        "from": honest_validator_address,
        "gas": TX_GAS_CONFIRM,
    })

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError("Honest confirmation failed although A_k matched expected A_k.")
else:
    print("[ACTION] No confirmation transaction was sent before the deadline.")


ann_after_validation = test_contract.functions.getAnnouncement(
    int(TEST_COMMUNITY_ID)
).call()

confirm_count_after_validation = int(ann_after_validation[2])
finalized_after_validation = bool(ann_after_validation[4])

print("[STATE AFTER VALIDATION]")
print("confirm_count:", confirm_count_after_validation)
print("finalized    :", finalized_after_validation)


# =====================================================================================
# Advance blockchain time until the confirmation window expires
# =====================================================================================

print("\n[STEP 6] Advance Ganache time until deadline passes...")
print(f"[INFO] Advancing blockchain time by {WAIT_SECONDS} seconds...")

advance_ganache_time(
    w3=w3,
    seconds=WAIT_SECONDS,
    owner_address=owner_address
)

current_block = w3.eth.get_block("latest")
current_block_timestamp = int(current_block["timestamp"])

timing_after_wait = test_contract.functions.getAnnouncementTiming(
    int(TEST_COMMUNITY_ID)
).call()

ann_after_wait = test_contract.functions.getAnnouncement(
    int(TEST_COMMUNITY_ID)
).call()

announced_at_after = int(timing_after_wait[0])
deadline_after = int(timing_after_wait[1])
expired_after_wait = bool(timing_after_wait[2])

confirm_count_after_wait = int(ann_after_wait[2])
exists_after_wait = bool(ann_after_wait[3])
finalized_after_wait = bool(ann_after_wait[4])

insufficient_confirmations_after_wait = (
    int(confirm_count_after_wait) < int(required_confirmations)
)

print("[STATE AFTER TIME ADVANCE]")
print("latest block timestamp :", current_block_timestamp)
print("exists                 :", exists_after_wait)
print("finalized              :", finalized_after_wait)
print("confirm_count          :", confirm_count_after_wait)
print("required_confirmations :", required_confirmations)
print("insufficient_confirmations:", insufficient_confirmations_after_wait)
print("expired                :", expired_after_wait)
print("announced_at           :", announced_at_after)
print("deadline               :", deadline_after)


# =====================================================================================
# Try confirmAk after deadline
# =====================================================================================

print("\n[STEP 7] Try confirmAk after deadline...")

late_confirm_rejected = False
late_confirm_error = None
late_confirm_receipt_status = None

try:
    tx_hash = test_contract.functions.confirmAk(
        int(TEST_COMMUNITY_ID),
        int(ANNOUNCED_AK)
    ).transact({
        "from": honest_validator_address,
        "gas": TX_GAS_CONFIRM,
    })

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    late_confirm_receipt_status = int(receipt.status)

    if receipt.status == 0:
        late_confirm_rejected = True
        print("[EXPECTED] Late confirmAk transaction reverted after deadline.")
    else:
        late_confirm_rejected = False
        print("[UNEXPECTED] Late confirmAk succeeded after deadline.")

except Exception as e:
    late_confirm_rejected = True
    late_confirm_error = str(e)
    print("[EXPECTED] Late confirmAk was rejected after deadline.")
    print("Reason preview:", late_confirm_error[:300])


# =====================================================================================
# Final result
# =====================================================================================

expected_result = "PASS" if (
    ak_matches_expected is False
    and honest_nodes_decision == "SKIP_CONFIRMATION"
    and expired_before_wait is False
    and expired_after_wait is True
    and insufficient_confirmations_after_wait is True
    and finalized_after_wait is False
    and late_confirm_rejected is True
) else "CHECK_NEEDED"

summary = pd.DataFrame([
    {
        "test_contract_address": test_contract_address,
        "community_id": int(TEST_COMMUNITY_ID),
        "expected_A_k": int(EXPECTED_AK),
        "announced_A_k": int(ANNOUNCED_AK),
        "ak_matches_expected": bool(ak_matches_expected),
        "honest_nodes_decision": honest_nodes_decision,
        "window_seconds": int(TEST_CONFIRMATION_WINDOW_SECONDS),
        "wait_seconds": int(WAIT_SECONDS),
        "required_confirmations": int(required_confirmations),
        "confirm_count_before_wait": int(confirm_count_before),
        "confirm_count_after_validation": int(confirm_count_after_validation),
        "confirm_count_after_wait": int(confirm_count_after_wait),
        "insufficient_confirmations_after_wait": bool(insufficient_confirmations_after_wait),
        "expired_before_wait": bool(expired_before_wait),
        "expired_after_wait": bool(expired_after_wait),
        "latest_block_timestamp_after_wait": int(current_block_timestamp),
        "deadline_after_wait": int(deadline_after),
        "finalized_before_wait": bool(finalized_before),
        "finalized_after_validation": bool(finalized_after_validation),
        "finalized_after_wait": bool(finalized_after_wait),
        "late_confirm_rejected": bool(late_confirm_rejected),
        "late_confirm_receipt_status": late_confirm_receipt_status,
        "expected_result": expected_result,
    }
])

summary.to_csv(
    INVALID_TIMING_TEST_REPORT_OUT,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print("[INVALID ANNOUNCEMENT + TIMING TEST SUMMARY]")
print(summary.to_string(index=False))
print("[PATH] Test report:", INVALID_TIMING_TEST_REPORT_OUT)

summary